# CS781 Phase 6 — Abstractive Legal Summarization

Phase 1 weights → document-to-summary supervision → dev BERTScore/ROUGE → verified test summaries.
Follows the professor's starter workflow and §12 of `Capstone Document_edited.pdf`.
**Only two official files exist:** train and test. A fixed, grouped held-out split is derived from train.
No test references are inspected. Every test document receives a generated summary and original ID.

All four Phase 1 backbones are used for generation, including the full encoder–decoder's pretrained
decoder and language-model heads. No external pretrained summarizer or API is used.
The RoBERTa model loaded by BERTScore is **evaluation only** and never supplies training targets.
Defaults: six core experiments; ten optional profiles; eight training trials; six decoding comparisons.
Everything saves to **`MyDrive/cs781_phase6`**. All runtime code is embedded in this notebook.

## 1. Install dependencies

In [ ]:
%pip -q install "tokenizers>=0.20,<0.24" "huggingface_hub>=0.26,<1" "transformers>=4.45,<5" "bert-score==0.3.13" "rouge-score==0.1.2" "numpy>=1.26" "tqdm>=4.66" matplotlib sentencepiece

## 2. Drive, HF token and GPU

In [ ]:
import os, json, math, random, time, zipfile, hashlib, gc, copy, getpass, platform
from pathlib import Path
from contextlib import nullcontext
from collections import Counter
from bisect import bisect_left, bisect_right
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from tokenizers import ByteLevelBPETokenizer

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive')
else:
    BASE_DIR = Path.cwd()

PHASE1_DIR = BASE_DIR / 'cs781_phase1' if IN_COLAB else BASE_DIR
PROJECT_DIR = BASE_DIR / 'cs781_phase6'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(PROJECT_DIR / 'hf_cache')
from huggingface_hub import hf_hub_download, login

HF_TOKEN = os.environ.get('HF_TOKEN')
if IN_COLAB and not HF_TOKEN:
    try:
        HF_TOKEN = userdata.get('HF')
    except Exception:
        pass
def validate_hf_token(value):
    # Never print or include the credential in an error message.
    if not isinstance(value, str):
        raise TypeError('HF token must be a plain text string, not ' + type(value).__name__ + '.')
    value = value.strip()
    if not value.startswith('hf_'):
        raise ValueError('Paste only the Hugging Face personal access token starting with hf_.')
    return value

if HF_TOKEN:
    try:
        HF_TOKEN = validate_hf_token(HF_TOKEN)
    except (TypeError, ValueError) as error:
        print(str(error))
        HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Paste only your hf_ token (hidden): ')
HF_TOKEN = validate_hf_token(HF_TOKEN)
login(token=HF_TOKEN, add_to_git_credential=False)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_DTYPE = torch.bfloat16 if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported() else torch.float16
print('Device:', DEVICE, '| PyTorch:', torch.__version__)
if DEVICE.type == 'cuda':
    print(torch.cuda.get_device_name(0))
else:
    print('CPU is suitable for checks; select a Colab GPU before full training.')
print('Phase 1 input:', PHASE1_DIR)
print('Phase 6 output:', PROJECT_DIR)

# Atomic writes avoid leaving partially written JSON/checkpoints after interruption.
def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.' + __import__('uuid').uuid4().hex + '.tmp')
    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(temporary, path)

def save_torch(data, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.' + __import__('uuid').uuid4().hex + '.tmp')
    torch.save(data, temporary)
    os.replace(temporary, path)

def amp_context():
    return torch.autocast('cuda', dtype=AMP_DTYPE) if DEVICE.type == 'cuda' else nullcontext()

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(seed)
import shutil

## 3. All controls — change here

Start with `EXPERIMENT='encoder_full'`, `RUN_ALL=False`. `RUN_ALL=True` runs the six core profiles,
with mandatory smaller-student KD last. Select any profile individually for separate Colab sessions.
Use separate notebook copies, finish shared token preparation once, and never run the same experiment/tag
in two sessions. A new manual setting needs a new `RUN_TAG`; matching settings resume automatically.
Section 12 explains tuning, validation subsets, generation limits and individual experiment names.

In [ ]:
SEED = 42
EXPERIMENT = 'encoder_full'
RUN_ALL = False
RUN_EXTRA_APPROACHES = False
RUN_TUNING = False
RUN_DECODING_SEARCH = False
RUN_TAG = 'v1'
RESUME = True
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
VALIDATION_FRACTION = 0.05
EXCLUDE_TRAIN_TEST_OVERLAP = True
INVALID_TRAIN_ROW_POLICY = 'exclude'  # exclude unusable labeled rows with an audit, or error
MAX_INVALID_TRAIN_FRACTION = 0.05  # Stop if >5% are unusable; inspect schema before proceeding.
EPOCHS = 4
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
MAX_SOURCE_TOKENS = 640
MAX_TARGET_TOKENS = 256  # Reference budget includes its terminal EOS.
SOURCE_POLICY = 'head_tail'  # head, tail, head_tail, spread, sentence_mmr
ENCODER_CHUNK_SIZE = 0  # Native encoder: 0=joint source; e.g. 384=encode windows independently.
INSTRUCTION = 'Summarize the legal facts, arguments, reasoning and decision.\nDocument:\n'
SUMMARY_PREFIX = '\nLegal summary:\n'
LEARNING_RATE = None  # None uses the profile LR. Set a number for manual tuning under a NEW tag.
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.1
LABEL_SMOOTHING = 0.1
GRADIENT_CHECKPOINTING = True
EARLY_STOPPING_PATIENCE = 2
SAVE_EVERY_STEPS = 200
MAX_OPTIMIZER_STEPS = None  # Smoke test only; capped runs are marked incomplete.
RANK = 8
ALPHA = 16.0
BOTTLENECK_DIM = 64
PROMPT_TOKENS = 16
PARTIAL_LAYERS = 2
KD_WEIGHT = 0.5
KD_TEMPERATURE = 2.0
TEACHER_RUN_DIR = None  # A completed, larger Phase 6 teacher; None selects best eligible saved run.
GENERATION = {'num_beams': 2, 'max_new_tokens': 256, 'min_new_tokens': 24,
              'length_penalty': 1.0, 'repetition_penalty': 1.1, 'no_repeat_ngram_size': 3,
              'deduplicate_sentences': False, 'long_document_mode': 'select', 'map_reduce_max_chunks': 4}
# Fixed deterministic subset for epoch selection, not a substitute for final full validation.
EPOCH_VALIDATION_LIMIT = 128  # 0 evaluates the entire held-out split at every epoch.
FINAL_VALIDATION_LIMIT = 0  # Keep 0 for real runs: score ALL held-out examples after training.
EVAL_LOSS_LIMIT = 256  # 0=all held-out examples for teacher-forced loss diagnostics.
BERTSCORE_MODEL = 'roberta-large'  # EVALUATION ONLY. Never used by the summarizer or teacher.
BERTSCORE_LAYERS = 17
BERTSCORE_RESCALE = False
BERTSCORE_BATCH_SIZE = 8
BERTSCORE_DEVICE = 'cpu'  # 'cuda' is faster but shares GPU memory with the summarizer.
ROUGE_USE_STEMMER = True
PHASE1_HEADS = {'decoder_only': 12, 'encoder_decoder': 8, 'hybrid_prefixlm': 10, 'lightweight': 12}
CORE_EXPERIMENTS = {
    'encoder_full': {'architecture': 'encoder_decoder', 'method': 'full', 'lr': 2e-5},
    'decoder_lora': {'architecture': 'decoder_only', 'method': 'lora', 'lr': 2e-4},
    'hybrid_lorma': {'architecture': 'hybrid_prefixlm', 'method': 'lorma', 'lr': 2e-4},
    'encoder_adapters': {'architecture': 'encoder_decoder', 'method': 'houlsby', 'lr': 2e-4},
    'lightweight_prompt': {'architecture': 'lightweight', 'method': 'prompt', 'lr': 5e-3},
    'lightweight_kd': {'architecture': 'lightweight', 'method': 'kd', 'lr': 2e-5},
}
EXTRA_EXPERIMENTS = {
    'decoder_full': {'architecture': 'decoder_only', 'method': 'full', 'lr': 2e-5},
    'encoder_lora': {'architecture': 'encoder_decoder', 'method': 'lora', 'lr': 2e-4},
    'encoder_lorma': {'architecture': 'encoder_decoder', 'method': 'lorma', 'lr': 2e-4},
    'encoder_partial': {'architecture': 'encoder_decoder', 'method': 'partial', 'lr': 5e-5},
    'hybrid_lora': {'architecture': 'hybrid_prefixlm', 'method': 'lora', 'lr': 2e-4},
    'hybrid_adapters': {'architecture': 'hybrid_prefixlm', 'method': 'houlsby', 'lr': 2e-4},
    'lightweight_lorma': {'architecture': 'lightweight', 'method': 'lorma', 'lr': 2e-4},
    'lightweight_lora': {'architecture': 'lightweight', 'method': 'lora', 'lr': 2e-4},
    'lightweight_pfeiffer': {'architecture': 'lightweight', 'method': 'pfeiffer', 'lr': 2e-4},
    'lightweight_full': {'architecture': 'lightweight', 'method': 'full', 'lr': 2e-5},
}
EXPERIMENTS = {**CORE_EXPERIMENTS, **EXTRA_EXPERIMENTS}
TUNE_BASE_EXPERIMENT = 'encoder_full'
TUNING_TRIALS = [
    {'name': 'lr_low', 'overrides': {'lr': 1e-5}},
    {'name': 'long_source', 'overrides': {'max_source_tokens': 896}},
    {'name': 'long_target', 'overrides': {'max_target_tokens': 384}},
    {'name': 'spread', 'overrides': {'source_policy': 'spread'}},
    {'name': 'mmr_context', 'overrides': {'source_policy': 'sentence_mmr'}},
    {'name': 'head_only', 'overrides': {'source_policy': 'head'}},
    {'name': 'no_smoothing', 'overrides': {'label_smoothing': 0.0}},
    {'name': 'independent_encoder_chunks', 'overrides': {'encoder_chunk_size': 384}},
]
DECODING_CANDIDATES = [
    {'name': 'greedy', 'overrides': {'num_beams': 1}},
    {'name': 'beam3', 'overrides': {'num_beams': 3}},
    {'name': 'longer', 'overrides': {'max_new_tokens': 384}},
    {'name': 'shorter', 'overrides': {'max_new_tokens': 160}},
    {'name': 'deduplicate', 'overrides': {'deduplicate_sentences': True}},
    {'name': 'map_reduce', 'overrides': {'long_document_mode': 'map_reduce'}},
]
assert EXPERIMENT in EXPERIMENTS and TUNE_BASE_EXPERIMENT in EXPERIMENTS
assert 0 < VALIDATION_FRACTION < 1 and EPOCHS > 0 and BATCH_SIZE > 0 and GRAD_ACCUM_STEPS > 0
assert MAX_OPTIMIZER_STEPS is None or MAX_OPTIMIZER_STEPS > 0
assert 0 <= LABEL_SMOOTHING < 1 and 0 <= KD_WEIGHT <= 1 and KD_TEMPERATURE > 0
seed_all(SEED)

## 4. Download train/test, create a fixed held-out split and tokenize

No nonexistent dev file is requested. Duplicate source texts stay in the same split. Optional exact
train/test text overlap filtering removes training copies only; test rows remain. The split manifest
records every retained/excluded ID. Empty string sources get an UNK token. String-list documents and
summaries are joined in their original paragraph order. Missing/blank or unsupported training fields
are audited and excluded by default; no reference is invented. More than 5% unusable training rows
stops for schema inspection. Test rows are never skipped. `data_quality_report.json` and
`excluded_training_rows.json` give IDs, reasons and field types without copying document contents.
Shared binary token caches persist in Drive and stage to Colab local disk for fast reads.

In [ ]:
def parse_case_file(path):
    import ast, csv, io
    path=Path(path)
    with path.open('rb') as stream:
        signature=stream.read(4)
    encoding='utf-16' if signature.startswith((b'\xff\xfe',b'\xfe\xff')) else 'utf-8-sig'
    try:
        content=path.read_text(encoding=encoding)
    except UnicodeError as exc:
        raise ValueError('File is not UTF-8/UTF-16 text; possibly an incomplete or non-text download.') from exc
    stripped=content.strip()
    if not stripped:
        raise ValueError('Downloaded file is empty or whitespace-only.')
    if stripped.startswith('version https://git-lfs.github.com/spec/v1'):
        raise ValueError('Downloaded file is a Git LFS pointer, not the dataset contents.')
    if stripped.lower().startswith(('<!doctype html','<html','<?xml')):
        raise ValueError('Downloaded file is an HTML/XML response, not case data.')
    try:
        return json.loads(content), 'json'
    except json.JSONDecodeError as exc:
        json_error=f'JSON parse failed at line {exc.lineno}, column {exc.colno}.'
    # Newline-delimited objects are accepted only if every nonblank line is valid.
    lines=io.StringIO(content)
    first=next((line.strip() for line in lines if line.strip()),'')
    if first.startswith('{'):
        try:
            initial=json.loads(first)
            if isinstance(initial,dict):
                rows=[initial]
                for number,line in enumerate(lines,2):
                    if not line.strip(): continue
                    row=json.loads(line)
                    if not isinstance(row,dict): raise ValueError('JSON Lines must contain objects.')
                    rows.append(row)
                return rows,'jsonl'
        except (json.JSONDecodeError,ValueError):
            pass
    # Some exports are CSV despite their .json suffix. Require known columns.
    csv_reader=csv.reader(io.StringIO(content))
    try:
        header=next(csv_reader)
    except (StopIteration,csv.Error):
        header=[]
    if 'id' in header and ('document' in header or 'text' in header):
        csv.field_size_limit(min(__import__('sys').maxsize,2**31-1))
        reader=csv.DictReader(io.StringIO(content),strict=True)
        rows=[]
        try:
            for row in reader:
                if None in row or any(value is None for value in row.values()):
                    raise ValueError('CSV row has missing or extra columns.')
                rows.append(row)
        except csv.Error as exc:
            raise ValueError('Malformed CSV case export.') from exc
        return rows,'csv'
    # Safe parsing only; never execute downloaded Python code.
    if stripped.lstrip('[').lstrip().startswith(('{'+chr(39),'{"')):
        try:
            literal=ast.literal_eval(stripped)
            if isinstance(literal,(list,dict)):
                return literal,'python_literal'
        except (SyntaxError,ValueError,TypeError,RecursionError):
            pass
    raise ValueError(json_error+' Unrecognized or malformed case-file format; no records were skipped.')


FILES={'train':'Phase_6_Summ/Train/train.json','test':'Phase_6_Summ/Test/test.json'}
loading_report={}
def load_download(split):
    filename=FILES[split]
    for attempt in range(2):
        path=Path(hf_hub_download(repo_id=HF_DATASET_REPO,filename=filename,repo_type='dataset',token=HF_TOKEN,force_download=attempt==1))
        print('Reading',split,filename,path.stat().st_size,'bytes')
        try:
            raw,kind=parse_case_file(path)
            loading_report[split]={'format':kind,'bytes':path.stat().st_size,'attempt':attempt+1}
            save_json(loading_report,PROJECT_DIR/'dataset_loading_report.json')
            return raw
        except ValueError as error:
            loading_report[split]={'error':str(error),'bytes':path.stat().st_size,'attempt':attempt+1}
            save_json(loading_report,PROJECT_DIR/'dataset_loading_report.json')
            if attempt: raise ValueError(f'{split}: invalid dataset after fresh download: {error}') from error
            print('Retrying',split,'with a fresh download after:',error)

def original_key(value):
    if isinstance(value, bool) or not isinstance(value, (str, int)):
        raise ValueError('Document IDs must be strings or integers.')
    return str(value)


NORMALIZATION_AUDIT = {}
def normalize_summaries(raw, labeled, invalid_policy=None, max_invalid_fraction=None):
    split = 'train' if labeled else 'test'
    policy = (invalid_policy or INVALID_TRAIN_ROW_POLICY) if labeled else 'error'
    limit = MAX_INVALID_TRAIN_FRACTION if max_invalid_fraction is None else max_invalid_fraction
    if policy not in ('exclude', 'error') or not 0 <= limit <= 1:
        raise ValueError('Invalid training-row policy or fraction limit.')
    if isinstance(raw, dict) and ('document' in raw or 'text' in raw) and 'id' in raw:
        entries = [(None, raw)]
    elif isinstance(raw, dict) and isinstance(raw.get('data'), list):
        entries = [(None, row) for row in raw['data']]
    elif isinstance(raw, dict):
        entries = list(raw.items())
    elif isinstance(raw, list):
        entries = [(None, row) for row in raw]
    else:
        raise ValueError('Expected a JSON document list or ID-keyed mapping.')
    result = {}; invalid = []; normalized = Counter()
    def record_audit():
        NORMALIZATION_AUDIT[split] = {'input_rows': len(entries), 'usable_rows': len(result),
            'invalid_rows': len(invalid), 'policy': policy,
            'excluded_rows': len(invalid) if policy == 'exclude' else 0,
            'reason_counts': dict(Counter(reason for row in invalid for reason in row['reasons'])),
            'paragraph_lists_joined': dict(normalized), 'invalid_examples': invalid[:20]}
        save_json(NORMALIZATION_AUDIT, PROJECT_DIR / 'data_quality_report.json')
        if labeled:
            save_json(invalid if policy == 'exclude' else [], PROJECT_DIR / 'excluded_training_rows.json')
    for i, (outer, row) in enumerate(entries):
        if not labeled and isinstance(row, str):
            row = {'id': outer, 'document': row}
        if not isinstance(row, dict):
            raise ValueError(f'Invalid document at row {i}.')
        doc_id = row.get('id', outer); key = original_key(doc_id)
        if outer is not None and str(outer) != key:
            raise ValueError('Mapping key differs from original document ID.')
        text = row.get('document', row.get('text'))
        if isinstance(text, list) and all(isinstance(t, str) for t in text):
            text = '\n\n'.join(text)
            normalized['document'] += 1
        summary = row.get('summary') if labeled else None
        if labeled and isinstance(summary, list) and all(isinstance(t, str) for t in summary):
            summary = '\n\n'.join(summary)
            normalized['summary'] += 1
        reasons = []
        if not isinstance(text, str):
            reasons.append('missing_document' if text is None else 'unsupported_document_type')
        if labeled:
            if not isinstance(summary, str):
                reasons.append('missing_summary' if summary is None else 'unsupported_summary_type')
            elif not summary.strip():
                reasons.append('empty_summary')
        if reasons:
            invalid.append({'row_index': i, 'id': doc_id, 'reasons': reasons, 'available_fields': sorted(row),
                            'document_type': type(text).__name__, 'summary_type': type(summary).__name__,
                            'document_characters': len(text) if isinstance(text, str) else None,
                            'summary_characters': len(summary.strip()) if isinstance(summary, str) else None})
            if policy == 'error':
                record_audit()
                raise ValueError(f'{split} ID {key}: {", ".join(reasons)} '
                                 f'(document={type(text).__name__}, summary={type(summary).__name__}). '
                                 'See data_quality_report.json; test rows are never skipped.')
            continue
        internal = f'public:row:{i}' if labeled else key
        if internal in result:
            raise ValueError(f'Duplicate original test ID {key}. No document was overwritten.')
        result[internal] = {'id': doc_id, 'text': text, 'summary': summary}
    record_audit()
    if labeled and invalid and len(invalid) / max(1, len(entries)) > limit:
        raise ValueError(f'{len(invalid)}/{len(entries)} training rows are unusable, above MAX_INVALID_TRAIN_FRACTION={limit}. '
                         'Inspect data_quality_report.json for missing fields/types before changing this limit.')
    print(f'{split}: retained {len(result)}/{len(entries)} rows; unusable rows: {len(invalid)}; paragraph lists joined: {dict(normalized)}')
    for row in invalid[:5]:
        print('  Excluded training ID:', row['id'], '; reasons:', ', '.join(row['reasons']))
    return result


PUBLIC = normalize_summaries(load_download('train'), True)
TEST = normalize_summaries(load_download('test'), False)
if not PUBLIC or not TEST:
    raise ValueError('Both official files must be nonempty.')
def document_hash(row):
    return hashlib.sha256(row['text'].encode()).hexdigest()
test_hashes = {document_hash(row) for row in TEST.values() if row['text'].strip()}
excluded = [key for key, row in PUBLIC.items() if row['text'].strip() and document_hash(row) in test_hashes] if EXCLUDE_TRAIN_TEST_OVERLAP else []
excluded_set = set(excluded)
groups = {}
for key, row in PUBLIC.items():
    if key not in excluded_set:
        groups.setdefault(document_hash(row), []).append(key)
if len(groups) < 2:
    raise ValueError('Need at least two distinct training documents after overlap filtering.')
ordered = sorted(groups, key=lambda h: hashlib.sha256((str(SEED) + h).encode()).hexdigest())
count = max(1, min(len(ordered) - 1, round(len(ordered) * VALIDATION_FRACTION)))
validation_hashes = set(ordered[:count])
DATA = {'train': {}, 'dev': {}, 'test': TEST}
for key, row in PUBLIC.items():
    if key not in excluded_set:
        DATA['dev' if document_hash(row) in validation_hashes else 'train'][key] = row
assert not ({document_hash(row) for row in DATA['train'].values()} & {document_hash(row) for row in DATA['dev'].values()})
fingerprint = hashlib.sha256(json.dumps([SEED, VALIDATION_FRACTION, EXCLUDE_TRAIN_TEST_OVERLAP]).encode())
for split, rows in DATA.items():
    for key, row in rows.items():
        fingerprint.update(json.dumps([split, key, row], sort_keys=True, ensure_ascii=False).encode())
DATA_FINGERPRINT = fingerprint.hexdigest()
save_json({'fingerprint': DATA_FINGERPRINT, 'official_files': FILES, 'validation_seed': SEED,
           'validation_fraction': VALIDATION_FRACTION, 'public_instances': len(PUBLIC),
           'raw_training_instances': NORMALIZATION_AUDIT['train']['input_rows'],
           'excluded_invalid_training_instances': NORMALIZATION_AUDIT['train']['excluded_rows'],
           'counts': {s: len(d) for s, d in DATA.items()}, 'excluded_train_test_matches': len(excluded),
           'train_ids': list(DATA['train']), 'validation_ids': list(DATA['dev']),
           'excluded_ids': [{'key': key, 'id': PUBLIC[key]['id']} for key in excluded]}, PROJECT_DIR / 'split_manifest.json')
del PUBLIC
print('Fixed grouped split:', {split: len(rows) for split, rows in DATA.items()}, '; excluded test copies:', len(excluded))

TOKENIZER_DIR = PHASE1_DIR / 'bpe_tokenizer'
tok = ByteLevelBPETokenizer(str(TOKENIZER_DIR / 'vocab.json'), str(TOKENIZER_DIR / 'merges.txt'))
VOCAB_SIZE_ACTUAL = tok.get_vocab_size()
def special_id(names):
    for name in names:
        value = tok.token_to_id(name)
        if value is not None:
            return value
    raise ValueError('Missing Phase 1 special token: ' + repr(names))
PAD_ID = special_id(['<pad>', '[PAD]'])
BOS_ID = special_id(['<bos>', '<s>', '[BOS]'])
EOS_ID = special_id(['<eos>', '</s>', '[EOS]'])
UNK_ID = special_id(['<unk>', '[UNK]'])
TOKENIZER_HASH = hashlib.sha256((TOKENIZER_DIR / 'vocab.json').read_bytes() + (TOKENIZER_DIR / 'merges.txt').read_bytes()).hexdigest()
for filename in ['vocab.json', 'merges.txt']:
    (PROJECT_DIR / 'bpe_tokenizer').mkdir(exist_ok=True)
    shutil.copy2(TOKENIZER_DIR / filename, PROJECT_DIR / 'bpe_tokenizer' / filename)


class TokenStore:
    def __init__(self, path, index):
        self.data = np.memmap(path, mode='r', dtype='<i4') if path.stat().st_size else np.empty(0, dtype='<i4')
        self.index = index
    def __getitem__(self, key):
        offset, length = self.index[key]
        return self.data[offset:offset + length].tolist()


TOKENS = {}; REFERENCES = {}
for split, rows in DATA.items():
    for field, store in [('text', TOKENS), ('summary', REFERENCES)]:
        if split == 'test' and field == 'summary':
            continue
        stem = f'{split}_{field}_{DATA_FINGERPRINT[:16]}_{TOKENIZER_HASH[:12]}'
        binary = PROJECT_DIR / (stem + '.bin'); index_path = PROJECT_DIR / (stem + '.json')
        if not binary.is_file() or not index_path.is_file():
            temporary = binary.with_name(binary.name + '.' + __import__('uuid').uuid4().hex + '.tmp')
            index = {}; offset = 0
            with temporary.open('wb') as stream:
                for key, row in tqdm(rows.items(), desc='Tokenizing ' + split + '/' + field, unit='document'):
                    ids = np.asarray(tok.encode(row[field], add_special_tokens=False).ids, dtype='<i4')
                    stream.write(ids.tobytes()); index[key] = [offset, len(ids)]; offset += len(ids)
            os.replace(temporary, binary); save_json(index, index_path)
        index = json.loads(index_path.read_text())
        if set(index) != set(rows):
            raise ValueError('Token cache IDs disagree.')
        active = binary
        if IN_COLAB:
            staging = Path('/content/cs781_phase6_tokens'); staging.mkdir(exist_ok=True)
            active = staging / binary.name
            if not active.is_file() or active.stat().st_size != binary.stat().st_size:
                temporary = active.with_name(active.name + '.tmp')
                with binary.open('rb') as source, temporary.open('wb') as target:
                    with tqdm.wrapattr(source, 'read', total=binary.stat().st_size, desc='Staging ' + stem, bytes=True) as progress:
                        shutil.copyfileobj(progress, target, 8 * 1024 * 1024)
                os.replace(temporary, active)
        store[split] = TokenStore(active, index)


def select_source(text, ids, budget, policy):
    if len(ids) <= budget:
        return ids or [UNK_ID], 1.0
    if policy == 'head':
        return ids[:budget], budget / len(ids)
    if policy == 'tail':
        return ids[-budget:], budget / len(ids)
    if policy == 'head_tail':
        first = (budget + 1) // 2
        return ids[:first] + ids[-(budget - first):] if budget > first else ids[:budget], budget / len(ids)
    if policy == 'spread':
        windows = min(4, budget); lengths = [budget // windows + (i < budget % windows) for i in range(windows)]
        boundaries = np.linspace(0, len(ids), windows + 1).astype(int)
        starts = [boundaries[i] if i < windows - 1 else len(ids) - lengths[i] for i in range(windows)]
        return [token for start, size in zip(starts, lengths) for token in ids[start:start + size]], budget / len(ids)
    if policy == 'sentence_mmr':
        import re
        sentences = re.split(r'(?<=[.!?])\s+|\n+', text)
        words = [Counter(re.findall(r'[a-z]+', sentence.lower())) for sentence in sentences]
        centroid = sum(words, Counter()); norm = math.sqrt(sum(v * v for v in centroid.values())) or 1.0
        selected = []; chosen = []; used = 0
        candidates = []
        for i, (sentence, vector) in enumerate(zip(sentences, words)):
            tokens = tok.encode(sentence, add_special_tokens=False).ids
            if not tokens or len(tokens) > budget:
                continue
            relevance = sum(v * centroid[k] for k, v in vector.items()) / (norm * (math.sqrt(sum(v * v for v in vector.values())) or 1))
            candidates.append((i, tokens, set(vector), relevance))
        while candidates:
            ranked = [(0.7 * item[3] - 0.3 * max((len(item[2] & old) / max(1, len(item[2] | old)) for old in chosen), default=0), j)
                      for j, item in enumerate(candidates) if used + len(item[1]) <= budget]
            if not ranked:
                break
            _, j = max(ranked); item = candidates.pop(j); selected.append(item); chosen.append(item[2]); used += len(item[1])
        if not selected:
            return select_source(text, ids, budget, 'head_tail')
        return [token for item in sorted(selected) for token in item[1]], min(1.0, used / len(ids))
    raise ValueError('Unknown source selection policy.')

## 5. Phase 1 architecture definitions and generation/adaptation wrappers

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def build_rope_cache(seq_len, head_dim, device, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.einsum("i,j->ij", t, inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1)
    return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]


def apply_rope(q, k, cos, sin):
    q_ = (q * cos) + (rotate_half(q) * sin)
    k_ = (k * cos) + (rotate_half(k) * sin)
    return q_, k_


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, max_len):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.register_buffer("mask", torch.tril(torch.ones(max_len, max_len)).bool(), persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(~self.mask[:T, :T], float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class GPTBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff, max_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, max_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class DecoderOnlyGPT(nn.Module):
    """Architecture 1: classic decoder-only Transformer LM."""

    def __init__(self, vocab_size, n_layer, n_embd, n_head, d_ff, max_len=2048):
        super().__init__()
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(max_len, n_embd)
        self.blocks = nn.ModuleList([GPTBlock(n_embd, n_head, d_ff, max_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.tok_emb(input_ids) + self.pos_emb(pos)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=PAD_ID,
            )
        return loss, logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temperature=0.8, top_p=0.95):
        # Simplified, non-KV-cached generation (recomputes full forward pass each step).
        # Positions are re-based to 0 each call since context is re-clipped to max_len.
        self.eval()
        for _ in range(max_new_tokens):
            ids_cond = input_ids[:, -self.max_len:]
            _, logits = self(ids_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            keep_mask = torch.arange(sorted_probs.size(-1), device=probs.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(keep_mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_id = sorted_idx.gather(-1, next_sorted)
            input_ids = torch.cat([input_ids, next_id], dim=1)
        return input_ids

In [ ]:
class RelativePositionBias(nn.Module):
    def __init__(self, n_head, num_buckets=32, max_distance=128, bidirectional=True):
        super().__init__()
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(num_buckets, n_head)

    @staticmethod
    def _bucket(rel_pos, bidirectional, num_buckets, max_distance):
        ret = 0
        n = -rel_pos
        if bidirectional:
            num_buckets //= 2
            ret += (n < 0).long() * num_buckets
            n = torch.abs(n)
        else:
            n = torch.clamp(n, min=0)
        max_exact = num_buckets // 2
        is_small = n < max_exact
        val_large = max_exact + (
            torch.log(n.float() / max_exact + 1e-6) / math.log(max_distance / max_exact) * (num_buckets - max_exact)
        ).long()
        val_large = torch.clamp(val_large, max=num_buckets - 1)
        ret = ret + torch.where(is_small, n, val_large)
        return ret

    def forward(self, q_len, k_len, device):
        q_pos = torch.arange(q_len, device=device)[:, None]
        k_pos = torch.arange(k_len, device=device)[None, :]
        rel_pos = k_pos - q_pos
        buckets = self._bucket(rel_pos, self.bidirectional, self.num_buckets, self.max_distance)
        values = self.embedding(buckets)                    # (q_len, k_len, n_head)
        return values.permute(2, 0, 1).unsqueeze(0)          # (1, n_head, q_len, k_len)


class RelPosAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x_q, x_kv, bias=None, causal=False, key_padding_mask=None):
        B, Tq, C = x_q.shape
        Tk = x_kv.shape[1]
        q = self.q_proj(x_q).view(B, Tq, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x_kv).view(B, Tk, self.n_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, Tk, self.n_head, self.d_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if bias is not None:
            att = att + bias
        if causal:
            causal_mask = torch.tril(torch.ones(Tq, Tk, device=x_q.device)).bool()
            att = att.masked_fill(~causal_mask, float("-inf"))
        if key_padding_mask is not None:
            pad_keep = key_padding_mask[:, None, None, :].bool()   # (B,1,1,Tk), 1 = keep
            att = att.masked_fill(~pad_keep, float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, Tq, C)
        return self.o_proj(out)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = RelPosAttention(d_model, n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x, bias, key_padding_mask):
        h = self.ln1(x)
        x = x + self.attn(h, h, bias=bias, causal=False, key_padding_mask=key_padding_mask)
        x = x + self.ff(self.ln2(x))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = RelPosAttention(d_model, n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = RelPosAttention(d_model, n_head)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x, enc_out, self_bias, enc_padding_mask):
        h = self.ln1(x)
        x = x + self.self_attn(h, h, bias=self_bias, causal=True)
        x = x + self.cross_attn(self.ln2(x), enc_out, bias=None, causal=False, key_padding_mask=enc_padding_mask)
        x = x + self.ff(self.ln3(x))
        return x


class EncoderDecoderTransformer(nn.Module):
    """Architecture 2: T5-style encoder-decoder Transformer with relative position bias."""

    def __init__(self, vocab_size, n_enc_layer, n_dec_layer, d_model, n_head, d_ff):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.enc_layers = nn.ModuleList([EncoderLayer(d_model, n_head, d_ff) for _ in range(n_enc_layer)])
        self.dec_layers = nn.ModuleList([DecoderLayer(d_model, n_head, d_ff) for _ in range(n_dec_layer)])
        self.enc_rel_bias = RelativePositionBias(n_head, bidirectional=True)
        self.dec_rel_bias = RelativePositionBias(n_head, bidirectional=False)
        self.ln_enc_f = nn.LayerNorm(d_model)
        self.ln_dec_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def encode(self, src_ids, src_mask):
        x = self.tok_emb(src_ids)
        bias = self.enc_rel_bias(src_ids.size(1), src_ids.size(1), src_ids.device)
        for layer in self.enc_layers:
            x = layer(x, bias, src_mask)
        return self.ln_enc_f(x)

    def forward(self, src_ids, src_mask, tgt_in, tgt_out=None):
        enc_out = self.encode(src_ids, src_mask)
        x = self.tok_emb(tgt_in)
        bias = self.dec_rel_bias(tgt_in.size(1), tgt_in.size(1), tgt_in.device)
        for layer in self.dec_layers:
            x = layer(x, enc_out, bias, src_mask)
        x = self.ln_dec_f(x)
        logits = self.head(x)
        loss = None
        if tgt_out is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1), ignore_index=PAD_ID)
        return loss, logits

    @torch.no_grad()
    def generate(self, src_ids, src_mask, max_new_tokens, bos_id, eos_id, temperature=0.8, top_p=0.95):
        # Simplified, non-KV-cached autoregressive decoding.
        self.eval()
        enc_out = self.encode(src_ids, src_mask)
        B = src_ids.size(0)
        tgt = torch.full((B, 1), bos_id, dtype=torch.long, device=src_ids.device)
        for _ in range(max_new_tokens):
            x = self.tok_emb(tgt)
            bias = self.dec_rel_bias(tgt.size(1), tgt.size(1), tgt.device)
            h = x
            for layer in self.dec_layers:
                h = layer(h, enc_out, bias, src_mask)
            h = self.ln_dec_f(h)
            logits = self.head(h)[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            keep_mask = torch.arange(sorted_probs.size(-1), device=probs.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(keep_mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_id = sorted_idx.gather(-1, next_sorted)
            tgt = torch.cat([tgt, next_id], dim=1)
        return tgt

In [ ]:
class RoPEAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, attn_mask, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(~attn_mask, float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class HybridBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = RoPEAttention(d_model, n_head)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff)

    def forward(self, x, attn_mask, cos, sin):
        x = x + self.attn(self.norm1(x), attn_mask, cos, sin)
        x = x + self.ff(self.norm2(x))
        return x


def build_prefix_lm_mask(prompt_len, total_len, device):
    """prompt<->prompt: bidirectional. continuation->prompt: visible.
    continuation<->continuation: causal."""
    mask = torch.zeros(total_len, total_len, dtype=torch.bool, device=device)
    mask[:prompt_len, :prompt_len] = True
    mask[prompt_len:, :prompt_len] = True
    cont_len = total_len - prompt_len
    if cont_len > 0:
        mask[prompt_len:, prompt_len:] = torch.tril(torch.ones(cont_len, cont_len, device=device)).bool()
    return mask


class HybridPrefixLM(nn.Module):
    """Architecture 3: single-stack encoder/hybrid LM (RoPE, RMSNorm, SwiGLU,
    UniLM-style prefix-bidirectional + causal-continuation attention mask)."""

    def __init__(self, vocab_size, n_layer, d_model, n_head, d_ff, max_len=1536):
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([HybridBlock(d_model, n_head, d_ff) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, prompt_len, labels=None):
        B, T = input_ids.shape
        x = self.tok_emb(input_ids)
        cos, sin = build_rope_cache(T, self.d_model // self.n_head, input_ids.device)
        mask = build_prefix_lm_mask(prompt_len, T, input_ids.device).unsqueeze(0).unsqueeze(0)
        for blk in self.blocks:
            x = blk(x, mask, cos, sin)
        x = self.norm_f(x)
        logits = self.head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=PAD_ID,
            )
        return loss, logits

    @torch.no_grad()
    def generate(self, prompt_ids, max_new_tokens, temperature=0.8, top_p=0.95):
        self.eval()
        input_ids = prompt_ids
        prompt_len = prompt_ids.size(1)
        for _ in range(max_new_tokens):
            ids_cond = input_ids[:, -self.max_len:]
            cur_prompt_len = min(prompt_len, ids_cond.size(1))
            _, logits = self(ids_cond, cur_prompt_len)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            keep_mask = torch.arange(sorted_probs.size(-1), device=probs.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(keep_mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_id = sorted_idx.gather(-1, next_sorted)
            input_ids = torch.cat([input_ids, next_id], dim=1)
        return input_ids

In [ ]:
class GQAAttention(nn.Module):
    """Grouped-Query Attention: n_kv_head < n_head, reducing parameters and
    (at inference, with KV caching) memory -- the key efficiency trick here."""

    def __init__(self, d_model, n_head, n_kv_head):
        super().__init__()
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.d_head = d_model // n_head
        self.q_proj = nn.Linear(d_model, n_head * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * self.d_head, bias=False)
        self.o_proj = nn.Linear(n_head * self.d_head, d_model, bias=False)

    def forward(self, x, causal_mask, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin)
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(~causal_mask, float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out)


class LightweightBlock(nn.Module):
    def __init__(self, d_model, n_head, n_kv_head, d_ff):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_head, n_kv_head)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff)

    def forward(self, x, mask, cos, sin):
        x = x + self.attn(self.norm1(x), mask, cos, sin)
        x = x + self.ff(self.norm2(x))
        return x


class LightweightLlama(nn.Module):
    """Architecture 4: efficient/lightweight decoder-only Transformer
    (RoPE, RMSNorm, SwiGLU, Grouped-Query Attention)."""

    def __init__(self, vocab_size, n_layer, d_model, n_head, n_kv_head, d_ff, max_len=2048):
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([LightweightBlock(d_model, n_head, n_kv_head, d_ff) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.tok_emb(input_ids)
        cos, sin = build_rope_cache(T, self.d_model // self.n_head, input_ids.device)
        mask = torch.tril(torch.ones(T, T, device=input_ids.device)).bool().unsqueeze(0).unsqueeze(0)
        for blk in self.blocks:
            x = blk(x, mask, cos, sin)
        x = self.norm_f(x)
        logits = self.head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=PAD_ID,
            )
        return loss, logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temperature=0.8, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            ids_cond = input_ids[:, -self.max_len:]
            _, logits = self(ids_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            keep_mask = torch.arange(sorted_probs.size(-1), device=probs.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(keep_mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_id = sorted_idx.gather(-1, next_sorted)
            input_ids = torch.cat([input_ids, next_id], dim=1)
        return input_ids

In [ ]:
def torch_load(path):
    return torch.load(path, map_location='cpu', weights_only=True)

def infer_phase1_config(architecture, state):
    vocab, dim = state['tok_emb.weight'].shape
    if vocab != VOCAB_SIZE_ACTUAL:
        raise ValueError(f'Phase 1 vocabulary {vocab} != tokenizer {VOCAB_SIZE_ACTUAL}')
    config = {'vocab_size': vocab}
    def count_layers(prefix):
        return len({int(k.split('.')[1]) for k in state if k.startswith(prefix + '.')})
    heads = PHASE1_HEADS[architecture]
    if architecture == 'decoder_only':
        config.update(n_layer=count_layers('blocks'), n_embd=dim, n_head=heads,
                      d_ff=state['blocks.0.mlp.0.weight'].shape[0], max_len=state['pos_emb.weight'].shape[0])
    elif architecture == 'encoder_decoder':
        stored_heads = state['enc_rel_bias.embedding.weight'].shape[1]
        if heads != stored_heads:
            raise ValueError('Encoder head count differs from stored relative-position bias.')
        config.update(n_enc_layer=count_layers('enc_layers'), n_dec_layer=count_layers('dec_layers'),
                      d_model=dim, n_head=heads, d_ff=state['enc_layers.0.ff.0.weight'].shape[0])
    elif architecture == 'hybrid_prefixlm':
        config.update(n_layer=count_layers('blocks'), d_model=dim, n_head=heads,
                      d_ff=state['blocks.0.ff.w1.weight'].shape[0], max_len=1152)
    else:
        head_dim = dim // heads
        config.update(n_layer=count_layers('blocks'), d_model=dim, n_head=heads,
                      n_kv_head=state['blocks.0.attn.k_proj.weight'].shape[0] // head_dim,
                      d_ff=state['blocks.0.ff.w1.weight'].shape[0], max_len=1024)
    if dim % heads:
        raise ValueError('Invalid Phase 1 attention head count.')
    return config

MODEL_CLASSES = {'decoder_only': DecoderOnlyGPT, 'encoder_decoder': EncoderDecoderTransformer,
                 'hybrid_prefixlm': HybridPrefixLM, 'lightweight': LightweightLlama}

In [ ]:
class LoRAProjection(nn.Module):
    def __init__(self, base, rank, alpha, multiplicative=False):
        super().__init__(); self.base = base; base.requires_grad_(False)
        self.multiplicative = multiplicative; self.scale = alpha / rank
        self.A = nn.Linear(base.out_features if multiplicative else base.in_features, rank, bias=False)
        self.B = nn.Linear(rank, base.out_features, bias=False)
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5)); nn.init.zeros_(self.B.weight)
    def forward(self, x):
        wx = F.linear(x, self.base.weight, None)
        update = self.B(self.A(wx if self.multiplicative else x)) * self.scale
        return wx + update if self.base.bias is None else wx + update + self.base.bias


class Bottleneck(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__(); self.norm = nn.LayerNorm(dim); self.down = nn.Linear(dim, hidden); self.up = nn.Linear(hidden, dim)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        return x + self.up(F.gelu(self.down(self.norm(x))))


def load_generative_phase1(architecture):
    path = PHASE1_DIR / ('run_' + architecture) / 'final_model.pt'
    if not path.is_file():
        raise FileNotFoundError('Missing Phase 1 checkpoint: ' + str(path))
    state = torch_load(path); config = infer_phase1_config(architecture, state)
    model = MODEL_CLASSES[architecture](**config)
    model.load_state_dict(state, strict=True)
    del state; gc.collect()
    return model, config


class LegalSummarizer(nn.Module):
    def __init__(self, backbone, phase1_config, cfg):
        super().__init__(); self.backbone = backbone; self.cfg = cfg
        self.architecture = cfg['spec']['architecture']; self.phase1_config = phase1_config
        self.max_context = phase1_config.get('max_len', 4096)
        method = cfg['spec']['method']; self.selected_layers = None
        if method == 'kd':
            self.selected_layers = list(range(0, len(backbone.blocks), 2))
            backbone.blocks = nn.ModuleList([backbone.blocks[i] for i in self.selected_layers])
        backbone.requires_grad_(method in ('full', 'kd'))
        if method == 'partial':
            collections = [backbone.enc_layers, backbone.dec_layers] if self.architecture == 'encoder_decoder' else [backbone.blocks]
            for layers in collections:
                for layer in list(layers)[-cfg['partial_layers']:]: layer.requires_grad_(True)
            for name, module in backbone.named_modules():
                if name in ['ln_f', 'norm_f', 'ln_enc_f', 'ln_dec_f']: module.requires_grad_(True)
        if method in ('lora', 'lorma'):
            for name, child in list(backbone.named_modules()):
                if isinstance(child, nn.Linear) and any(part in name for part in ['attn.', 'self_attn.', 'cross_attn.']):
                    parent, leaf = name.rsplit('.', 1)
                    setattr(backbone.get_submodule(parent), leaf, LoRAProjection(child, cfg['rank'], cfg['alpha'], method == 'lorma'))
        dim = backbone.tok_emb.embedding_dim
        self.adapters = nn.ModuleDict()
        if method in ('houlsby', 'pfeiffer'):
            collections = [('enc', backbone.enc_layers), ('dec', backbone.dec_layers)] if self.architecture == 'encoder_decoder' else [('stack', backbone.blocks)]
            for prefix, layers in collections:
                for i in range(len(layers)):
                    # Houlsby placement: after attention and feed-forward residuals.
                    # Pfeiffer-style comparison: one feed-forward/output bottleneck per block.
                    sites = ['attn', 'ff'] if method == 'houlsby' else ['ff']
                    if prefix == 'dec' and method == 'houlsby': sites.append('cross')
                    for site in sites: self.adapters[f'{prefix}_{i}_{site}'] = Bottleneck(dim, cfg['bottleneck_dim'])
        self.soft_prompt = None
        if method == 'prompt':
            self.soft_prompt = nn.Parameter(torch.empty(cfg['prompt_tokens'], dim))
            nn.init.normal_(self.soft_prompt, std=0.02)
        if not any(p.requires_grad for p in self.parameters()):
            raise ValueError('No trainable parameters in this profile.')

    def adapt(self, x, name):
        return self.adapters[name](x) if name in self.adapters else x

    @staticmethod
    def attend(attn, query, memory=None, keep=None, bias=None, past=None, rope=None, static_kv=None):
        memory = query if memory is None else memory
        batch, length, dim = query.shape; heads = attn.n_head; head_dim = attn.d_head
        if static_kv is not None:
            q = attn.q_proj(query); kv_heads = heads
            k, v = static_kv
        elif hasattr(attn, 'qkv'):
            q, k, v = attn.qkv(query).split(dim, dim=-1)
            kv_heads = heads
        else:
            q = attn.q_proj(query); k = attn.k_proj(memory); v = attn.v_proj(memory)
            kv_heads = getattr(attn, 'n_kv_head', heads)
        q = q.view(batch, length, heads, head_dim).transpose(1, 2)
        if static_kv is None:
            k = k.view(batch, -1, kv_heads, head_dim).transpose(1, 2)
            v = v.view(batch, -1, kv_heads, head_dim).transpose(1, 2)
        if rope is not None: q, k = apply_rope(q, k, *(value.to(q.dtype) for value in rope))
        if past is not None: k = torch.cat([past[0], k], dim=2); v = torch.cat([past[1], v], dim=2)
        cache = (k, v)
        if kv_heads != heads:
            k = k.repeat_interleave(heads // kv_heads, dim=1); v = v.repeat_interleave(heads // kv_heads, dim=1)
        mask = keep
        if bias is not None:
            mask = bias if keep is None else bias.masked_fill(~keep, float('-inf'))
        output = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0)
        output = output.transpose(1, 2).reshape(batch, length, dim)
        projection = attn.proj if hasattr(attn, 'proj') else attn.o_proj
        return projection(output), cache

    def encode(self, source, source_mask):
        b = self.backbone; parts = []
        size = self.cfg['encoder_chunk_size'] or source.shape[1]
        for start in range(0, source.shape[1], size):
            ids = source[:, start:start + size]; keep = source_mask[:, start:start + size]
            x = b.tok_emb(ids)
            # A fully padded chunk for a shorter batch member must still have one finite attention key.
            safe = keep.clone(); safe[:, 0] = True
            bias = b.enc_rel_bias(ids.shape[1], ids.shape[1], ids.device)
            for i, layer in enumerate(b.enc_layers):
                def block(h, layer=layer, i=i, safe=safe, bias=bias):
                    norm = layer.ln1(h)
                    update, _ = self.attend(layer.attn, norm, keep=safe[:, None, None, :], bias=bias)
                    h = self.adapt(h + update, f'enc_{i}_attn')
                    return self.adapt(h + layer.ff(layer.ln2(h)), f'enc_{i}_ff')
                x = checkpoint(block, x, use_reentrant=False) if self.training and self.cfg['gradient_checkpointing'] else block(x)
            parts.append(b.ln_enc_f(x))
        return torch.cat(parts, dim=1), source_mask

    def hidden(self, ids, keep, prefix_lengths=None, past=None, encoder=None, use_cache=False):
        b = self.backbone; architecture = self.architecture
        offset = past[0][0].shape[2] if past else 0
        x = b.tok_emb(ids)
        if self.soft_prompt is not None and offset == 0:
            x = torch.cat([self.soft_prompt[None].expand(len(ids), -1, -1), x], dim=1)
            keep = torch.cat([torch.ones(len(ids), len(self.soft_prompt), dtype=torch.bool, device=ids.device), keep], dim=1)
            if prefix_lengths is not None: prefix_lengths = prefix_lengths + len(self.soft_prompt)
        length = x.shape[1]; total = offset + length
        qpos = torch.arange(offset, total, device=ids.device)
        kpos = torch.arange(total, device=ids.device)
        causal = kpos[None, :] <= qpos[:, None]
        mask = causal[None, None] & keep[:, None, None, :]
        if architecture == 'hybrid_prefixlm' and offset == 0:
            prompt = prefix_lengths[:, None, None]
            bidirectional = (qpos[None, :, None] < prompt) & (kpos[None, None, :] < prompt)
            mask = (causal[None] | bidirectional)[:, None] & keep[:, None, None, :]
        if architecture == 'decoder_only': x = x + b.pos_emb(qpos)[None]
        rotary = None
        if architecture in ('hybrid_prefixlm', 'lightweight'):
            cos, sin = build_rope_cache(total, b.tok_emb.embedding_dim // b.n_head, ids.device)
            rotary = (cos[:, :, offset:total, :], sin[:, :, offset:total, :])
        caches = []
        layers = b.dec_layers if architecture == 'encoder_decoder' else b.blocks
        bias = b.dec_rel_bias(total, total, ids.device)[:, :, offset:total, :] if architecture == 'encoder_decoder' else None
        for i, layer in enumerate(layers):
            old = past[i] if past else None
            def block(h, layer=layer, i=i, old=old):
                if architecture == 'encoder_decoder':
                    update, cache = self.attend(layer.self_attn, layer.ln1(h), keep=mask, bias=bias, past=old[:2] if old else None)
                    h = self.adapt(h + update, f'dec_{i}_attn')
                    update, cross_cache = self.attend(layer.cross_attn, layer.ln2(h), memory=encoder[0],
                                                      keep=encoder[1][:, None, None, :], static_kv=old[2:] if old else None)
                    if use_cache: cache = (*cache, *cross_cache)
                    h = self.adapt(h + update, f'dec_{i}_cross')
                    h = self.adapt(h + layer.ff(layer.ln3(h)), f'dec_{i}_ff')
                elif architecture == 'decoder_only':
                    update, cache = self.attend(layer.attn, layer.ln1(h), keep=mask, past=old)
                    h = self.adapt(h + update, f'stack_{i}_attn')
                    h = self.adapt(h + layer.mlp(layer.ln2(h)), f'stack_{i}_ff')
                else:
                    update, cache = self.attend(layer.attn, layer.norm1(h), keep=mask, past=old, rope=rotary)
                    h = self.adapt(h + update, f'stack_{i}_attn')
                    h = self.adapt(h + layer.ff(layer.norm2(h)), f'stack_{i}_ff')
                return (h, cache) if use_cache else h
            if self.training and self.cfg['gradient_checkpointing'] and not use_cache:
                x = checkpoint(block, x, use_reentrant=False)
            elif use_cache:
                x, cache = block(x); caches.append(cache)
            else:
                x = block(x)
        final_norm = b.ln_dec_f if architecture == 'encoder_decoder' else (b.ln_f if architecture == 'decoder_only' else b.norm_f)
        return final_norm(x), caches if use_cache else None

    def forward(self, batch):
        encoder = self.encode(batch['source'], batch['source_mask']) if self.architecture == 'encoder_decoder' else None
        hidden, _ = self.hidden(batch['input_ids'], batch['input_mask'], batch['prefix_lengths'], encoder=encoder)
        prompt = len(self.soft_prompt) if self.soft_prompt is not None else 0
        indices = batch['target_positions'] + prompt
        selected = hidden.gather(1, indices[:, :, None].expand(-1, -1, hidden.shape[-1]))
        return self.backbone.head(selected)


def make_model(cfg):
    backbone, phase1 = load_generative_phase1(cfg['spec']['architecture'])
    model = LegalSummarizer(backbone, phase1, cfg)
    return model, {'phase1_config': phase1, 'selected_phase1_layers': model.selected_layers,
                   'total_parameters': sum(p.numel() for p in model.parameters()),
                   'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad)}


def pack_batch(split, keys, cfg, targets=None):
    architecture = cfg['spec']['architecture']
    instruction = tok.encode(cfg['instruction'], add_special_tokens=False).ids
    separator = tok.encode(cfg['summary_prefix'], add_special_tokens=False).ids
    targets = targets or [(REFERENCES[split][key][:cfg['max_target_tokens'] - 1] + [EOS_ID]) for key in keys]
    longest_target = max(map(len, targets)); sequences = []; prefixes = []; sources = []; positions = []; coverage = []
    # Native relative-position encoder has a separate source budget. Decoder families share one context.
    soft = cfg['prompt_tokens'] if cfg['spec']['method'] == 'prompt' else 0
    context = cfg['context_limit']
    source_budget = cfg['max_source_tokens'] if architecture == 'encoder_decoder' else min(cfg['max_source_tokens'], context - soft - len(instruction) - len(separator) - longest_target - 2)
    if source_budget < 1: raise ValueError('Target/prompt leaves no source context; reduce lengths.')
    for key, target in zip(keys, targets):
        selected, ratio = select_source(DATA[split][key]['text'], TOKENS[split][key], source_budget, cfg['source_policy'])
        source = [BOS_ID] + selected + [EOS_ID]; sources.append(source); coverage.append(ratio)
        if architecture == 'encoder_decoder':
            sequence = [BOS_ID] + target[:-1]; prefix = 1
        else:
            prompt = [BOS_ID] + instruction + selected + separator
            prefix = len(prompt); sequence = prompt + target[:-1]
        sequences.append(sequence); prefixes.append(prefix)
        positions.append([prefix - 1 + j for j in range(len(target))])
    def pad(rows, value, dtype=torch.long):
        tensor = torch.full((len(rows), max(map(len, rows))), value, dtype=dtype)
        for i, row in enumerate(rows): tensor[i, :len(row)] = torch.tensor(row, dtype=dtype)
        return tensor.to(DEVICE)
    return {'keys': keys, 'targets_list': targets, 'input_ids': pad(sequences, PAD_ID),
            'input_mask': pad([[True] * len(row) for row in sequences], False, torch.bool),
            'prefix_lengths': torch.tensor(prefixes, device=DEVICE), 'source': pad(sources, PAD_ID),
            'source_mask': pad([[True] * len(row) for row in sources], False, torch.bool),
            'target_positions': pad(positions, 0), 'labels': pad(targets, -100), 'coverage': coverage}

## 6. Cached beam decoding and submission validation

The source prefix is computed once; each decoder step reuses past attention keys/values. Native
encoder states are reused throughout generation. Greedy/beam search are deterministic. Length,
repetition penalties and n-gram blocking are dev-tunable. Optional map–reduce generates intermediate
chunk summaries before a final generated summary; its extra inference cost is recorded by runtime.

In [ ]:
def clean_summary(text, deduplicate=False):
    import re
    text = re.sub(r'\s+', ' ', text).strip()
    if deduplicate:
        seen = set(); kept = []
        for sentence in re.split(r'(?<=[.!?])\s+', text):
            key = re.sub(r'\W+', '', sentence.lower())
            if key and key not in seen: kept.append(sentence); seen.add(key)
        text = ' '.join(kept)
    return text


def block_repeats(logits, sequences, policy):
    for row, sequence in enumerate(sequences):
        penalty = policy['repetition_penalty']
        if sequence and penalty != 1:
            indices = torch.tensor(sorted(set(sequence)), device=logits.device)
            values = logits[row, indices]
            logits[row, indices] = torch.where(values < 0, values * penalty, values / penalty)
        n = policy['no_repeat_ngram_size']
        if n > 0 and len(sequence) >= n - 1:
            prefix = tuple(sequence[-(n - 1):]) if n > 1 else ()
            banned = {sequence[j + n - 1] for j in range(len(sequence) - n + 1)
                      if tuple(sequence[j:j + n - 1]) == prefix}
            if banned: logits[row, list(banned)] = float('-inf')
    return logits


@torch.no_grad()
def generate_tokens(model, source_ids, cfg, policy):
    architecture = cfg['spec']['architecture']; model.eval()
    soft = len(model.soft_prompt) if model.soft_prompt is not None else 0
    instruction = tok.encode(cfg['instruction'], add_special_tokens=False).ids
    separator = tok.encode(cfg['summary_prefix'], add_special_tokens=False).ids
    budget = cfg['max_source_tokens'] if architecture == 'encoder_decoder' else min(
        cfg['max_source_tokens'], model.max_context - policy['max_new_tokens'] - soft - len(instruction) - len(separator) - 1)
    if budget < 1: raise ValueError('Generation length leaves no source context.')
    # Rebudget an already selected context by retaining its beginning/end.
    if len(source_ids) > budget: source_ids, _ = select_source('', source_ids, budget, 'head_tail')
    source_ids = source_ids or [UNK_ID]
    encoder = None
    if architecture == 'encoder_decoder':
        src = torch.tensor([[BOS_ID] + source_ids + [EOS_ID]], device=DEVICE)
        with amp_context(): encoder = model.encode(src, torch.ones_like(src, dtype=torch.bool))
        prompt = [BOS_ID]
    else:
        prompt = [BOS_ID] + instruction + source_ids + separator
    ids = torch.tensor([prompt], device=DEVICE)
    with amp_context():
        hidden, cache = model.hidden(ids, torch.ones_like(ids, dtype=torch.bool),
                                     torch.tensor([len(prompt)], device=DEVICE), encoder=encoder, use_cache=True)
        next_logits = model.backbone.head(hidden[:, -1]).float()
    sequences = [[]]; scores = torch.zeros(1, device=DEVICE); finished = []; beams = policy['num_beams']
    for step in range(policy['max_new_tokens']):
        logits = block_repeats(next_logits.clone(), sequences, policy)
        blocked = [PAD_ID, BOS_ID, UNK_ID]
        sep_id = tok.token_to_id('<sep>')
        if sep_id is not None: blocked.append(sep_id)
        logits[:, blocked] = float('-inf')
        if step < policy['min_new_tokens']: logits[:, EOS_ID] = float('-inf')
        if not torch.isfinite(logits).any(dim=1).all(): raise FloatingPointError('No finite decoding candidates.')
        log_probs = torch.log_softmax(logits, dim=-1)
        candidate = scores[:, None] + log_probs
        values, indices = candidate.flatten().topk(min(candidate.numel(), beams * 2))
        alive = []; origins = []; tokens = []; next_scores = []
        for value, index in zip(values.tolist(), indices.tolist()):
            origin, token = divmod(index, logits.shape[-1]); sequence = sequences[origin] + [token]
            if token == EOS_ID:
                finished.append((value / max(1, len(sequence)) ** policy['length_penalty'], sequence))
            elif len(alive) < beams:
                alive.append(sequence); origins.append(origin); tokens.append(token); next_scores.append(value)
        # For greedy decoding EOS terminates immediately, rather than taking the second-best token.
        if beams == 1 and int(indices[0] % logits.shape[-1]) == EOS_ID: break
        if not alive: break
        sequences = alive; scores = torch.tensor(next_scores, device=DEVICE)
        order = torch.tensor(origins, device=DEVICE)
        cache = [tuple(value.index_select(0, order) for value in layer) for layer in cache]
        if encoder is not None: encoder = (encoder[0].index_select(0, order), encoder[1].index_select(0, order))
        if step + 1 == policy['max_new_tokens']: break
        ids = torch.tensor(tokens, device=DEVICE)[:, None]
        keep = torch.ones(len(tokens), cache[0][0].shape[2] + 1, dtype=torch.bool, device=DEVICE)
        with amp_context():
            hidden, cache = model.hidden(ids, keep, past=cache, encoder=encoder, use_cache=True)
            next_logits = model.backbone.head(hidden[:, -1]).float()
    for score, sequence in zip(scores.tolist(), sequences):
        if sequence: finished.append((score / len(sequence) ** policy['length_penalty'], sequence))
    if not finished: raise RuntimeError('Decoding produced no candidates.')
    best = max(finished, key=lambda item: item[0])[1]
    return [token for token in best if token not in (EOS_ID, PAD_ID, BOS_ID)]


@torch.no_grad()
def generate_summary(model, split, key, cfg, policy):
    ids = TOKENS[split][key]
    selected, _ = select_source(DATA[split][key]['text'], ids, cfg['max_source_tokens'], cfg['source_policy'])
    if policy['long_document_mode'] == 'map_reduce' and len(ids) > cfg['max_source_tokens']:
        ranges = [(i, min(i + cfg['max_source_tokens'], len(ids))) for i in range(0, len(ids), cfg['max_source_tokens'])]
        if len(ranges) > policy['map_reduce_max_chunks']:
            chosen = np.linspace(0, len(ranges) - 1, policy['map_reduce_max_chunks']).round().astype(int)
            ranges = [ranges[i] for i in chosen]
        intermediate = []
        for start, stop in ranges: intermediate.extend(generate_tokens(model, ids[start:stop], cfg, policy))
        selected, _ = select_source(tok.decode(intermediate), intermediate, cfg['max_source_tokens'], cfg['source_policy'])
    generated = generate_tokens(model, selected, cfg, policy)
    summary = clean_summary(tok.decode(generated), policy['deduplicate_sentences'])
    if not summary:
        raise RuntimeError(f'Empty generated summary for {key}; inspect model/decoding before submission.')
    return summary


def generate_rows(model, split, keys, cfg, policy):
    return [{'id': DATA[split][key]['id'], 'summary': generate_summary(model, split, key, cfg, policy)}
            for key in tqdm(keys, desc='Generating ' + split + ' (KV cache)', unit='document')]


def validate_rows(rows, split, keys):
    if not isinstance(rows, list) or len(rows) != len(keys): raise ValueError('Expected one summary per document.')
    for row, key in zip(rows, keys):
        original = DATA[split][key]['id']
        if set(row) != {'id', 'summary'} or type(row['id']) is not type(original) or row['id'] != original:
            raise ValueError('Summary fields/order/original IDs disagree.')
        if not isinstance(row['summary'], str) or not row['summary'].strip():
            raise ValueError('Every generated summary must be nonempty text.')
    if split == 'test' and len({original_key(row['id']) for row in rows}) != len(rows):
        raise ValueError('Duplicate test ID.')
    return True


def save_submission(output, rows):
    keys = list(DATA['test']); validate_rows(rows, 'test', keys)
    save_json(rows, output / 'prediction.json')
    temporary = output / 'submission.zip.tmp'
    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED) as archive:
        archive.write(output / 'prediction.json', arcname='prediction.json')
    os.replace(temporary, output / 'submission.zip')
    with zipfile.ZipFile(output / 'submission.zip') as archive:
        assert archive.namelist() == ['prediction.json']
        saved = json.loads(archive.read('prediction.json')); validate_rows(saved, 'test', keys)
        assert saved == rows

## 7. Primary BERTScore F1 and secondary ROUGE-1/ROUGE-L F1

RoBERTa-large, layer 17, no IDF or baseline rescaling is the explicit default. Metric model/hash/settings
are saved. The PDF does not publish exact evaluator settings, so equality to its hidden scorer is not
claimed. ROUGE uses the specified Porter-stemming flag and mean per-document F1.
BERTScore model/tokenizer downloads are solely for scoring; they are never used by the summarizer.
No fallback silently substitutes ROUGE for the primary metric if BERTScore cannot load.

In [ ]:
def capture_rng():
    return {'python': random.getstate(), 'numpy': {
        'kind': np.random.get_state()[0], 'keys': np.random.get_state()[1].tolist(),
        'pos': np.random.get_state()[2], 'gauss': np.random.get_state()[3],
        'cached': np.random.get_state()[4]}, 'torch': torch.get_rng_state(),
        'cuda': torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else []}

In [ ]:
def restore_rng(state):
    random.setstate(state['python'])
    n = state['numpy']
    np.random.set_state((n['kind'], np.array(n['keys'], dtype=np.uint32), n['pos'], n['gauss'], n['cached']))
    torch.set_rng_state(state['torch'])
    if DEVICE.type == 'cuda' and state['cuda']:
        torch.cuda.set_rng_state_all(state['cuda'])

In [ ]:
def metric_config():
    return {'model': BERTSCORE_MODEL, 'layers': BERTSCORE_LAYERS, 'rescale': BERTSCORE_RESCALE,
            'idf': False, 'rouge_use_stemmer': ROUGE_USE_STEMMER}


SCORER = None
SCORER_SIGNATURE = None
def get_scorer():
    global SCORER, SCORER_SIGNATURE
    signature = (BERTSCORE_MODEL, BERTSCORE_LAYERS, BERTSCORE_RESCALE, BERTSCORE_DEVICE)
    if SCORER is None or SCORER_SIGNATURE != signature:
        SCORER = None
        gc.collect()
        from bert_score import BERTScorer
        print('Loading evaluation-only BERTScore model:', BERTSCORE_MODEL, 'on', BERTSCORE_DEVICE)
        rng = capture_rng()
        try:
            SCORER = BERTScorer(model_type=BERTSCORE_MODEL, num_layers=BERTSCORE_LAYERS, lang='en',
                                rescale_with_baseline=BERTSCORE_RESCALE, idf=False,
                                device=BERTSCORE_DEVICE, batch_size=BERTSCORE_BATCH_SIZE)
            SCORER_SIGNATURE = signature
        finally:
            restore_rng(rng)
    return SCORER


def score_summaries(rows, split, keys):
    from rouge_score import rouge_scorer
    validate_rows(rows, split, keys)
    references = [DATA[split][key]['summary'] for key in keys]
    if any(not isinstance(reference, str) or not reference.strip() for reference in references):
        raise ValueError('Reference summaries required; test scoring is not available.')
    predictions = [row['summary'] for row in rows]
    scorer = get_scorer()
    with torch.no_grad():
        precision, recall, f1 = scorer.score(predictions, references, batch_size=BERTSCORE_BATCH_SIZE, verbose=True)
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=ROUGE_USE_STEMMER)
    scores = [rouge.score(reference, prediction) for reference, prediction in
              tqdm(zip(references, predictions), total=len(keys), desc='ROUGE validation', unit='document')]
    result = {'bertscore_f1': float(f1.mean()), 'bertscore_precision': float(precision.mean()),
              'bertscore_recall': float(recall.mean()),
              'rouge1_f1': float(np.mean([row['rouge1'].fmeasure for row in scores])),
              'rougeL_f1': float(np.mean([row['rougeL'].fmeasure for row in scores])),
              'cases': len(keys), 'bert_score_hash': scorer.hash, 'metric_config': metric_config(),
              'mean_generated_words': float(np.mean([len(text.split()) for text in predictions])),
              'mean_reference_words': float(np.mean([len(text.split()) for text in references]))}
    assert all(math.isfinite(result[k]) for k in ['bertscore_f1', 'rouge1_f1', 'rougeL_f1'])
    return result


def show_scores(scores, title):
    print(title)
    print(f"BERTScore F1 [PRIMARY]: {scores['bertscore_f1']:.6f}; ROUGE-1 F1: {scores['rouge1_f1']:.6f}; ROUGE-L F1: {scores['rougeL_f1']:.6f}")
    print('Documents:', scores['cases'], '; mean generated/reference words:', scores['mean_generated_words'], '/', scores['mean_reference_words'])


def fixed_validation_keys(limit):
    keys = sorted(DATA['dev'], key=lambda key: hashlib.sha256((str(SEED) + key).encode()).hexdigest())
    return keys[:limit] if limit else keys


@torch.no_grad()
def validation_loss(model, cfg):
    model.eval(); keys = fixed_validation_keys(cfg['eval_loss_limit']); total = 0.0; tokens = 0
    for start in tqdm(range(0, len(keys), cfg['batch_size']), desc='Validation target loss', unit='batch'):
        batch = pack_batch('dev', keys[start:start + cfg['batch_size']], cfg)
        with amp_context(): logits = model(batch)
        count = int((batch['labels'] != -100).sum())
        value = F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), batch['labels'].reshape(-1), ignore_index=-100, reduction='sum')
        total += float(value); tokens += count
    return total / max(1, tokens)

## 8. Train/adapt, select on fixed validation, reload and independently verify

Only summary tokens contribute to supervised loss. Source/padding labels are masked. Native seq2seq
targets and decoder-only concatenated targets are shifted consistently. LoRA/LoRMA/adapter/prompt
parameters actually train. KD matches a larger frozen Phase 6 teacher's next-token distributions on
training references only, with a smaller student retaining alternating Phase 1 lightweight layers.

Epoch selection uses a fixed validation subset by default. Final evaluation scores **all** held-out
documents when `FINAL_VALIDATION_LIMIT=0`; subset and full scores are kept separate. Interrupted
training checkpoints include optimizer/scheduler/scaler/RNG state and the exact next batch.

In [ ]:
def base_config(name, overrides=None):
    cfg = {'implementation_version': 1, 'experiment': name, 'spec': copy.deepcopy(EXPERIMENTS[name]), 'run_tag': RUN_TAG, 'seed': SEED,
           'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'grad_accum': GRAD_ACCUM_STEPS,
           'max_source_tokens': MAX_SOURCE_TOKENS, 'max_target_tokens': MAX_TARGET_TOKENS,
           'source_policy': SOURCE_POLICY, 'encoder_chunk_size': ENCODER_CHUNK_SIZE,
           'instruction': INSTRUCTION, 'summary_prefix': SUMMARY_PREFIX, 'weight_decay': WEIGHT_DECAY,
           'warmup_fraction': WARMUP_FRACTION, 'label_smoothing': LABEL_SMOOTHING,
           'gradient_checkpointing': GRADIENT_CHECKPOINTING, 'patience': EARLY_STOPPING_PATIENCE,
           'rank': RANK, 'alpha': ALPHA, 'bottleneck_dim': BOTTLENECK_DIM, 'prompt_tokens': PROMPT_TOKENS,
           'partial_layers': PARTIAL_LAYERS, 'kd_weight': KD_WEIGHT, 'kd_temperature': KD_TEMPERATURE,
           'teacher_requested': TEACHER_RUN_DIR, 'generation': copy.deepcopy(GENERATION),
           'epoch_validation_limit': EPOCH_VALIDATION_LIMIT, 'final_validation_limit': FINAL_VALIDATION_LIMIT,
           'eval_loss_limit': EVAL_LOSS_LIMIT, 'metric_config': metric_config(),
           'data_fingerprint': DATA_FINGERPRINT, 'tokenizer_hash': TOKENIZER_HASH,
           'phase1_heads': PHASE1_HEADS, 'max_optimizer_steps': MAX_OPTIMIZER_STEPS}
    if LEARNING_RATE is not None: cfg['spec']['lr'] = LEARNING_RATE
    for key, value in (overrides or {}).items():
        if key == 'lr': cfg['spec']['lr'] = value
        elif key in cfg: cfg[key] = value
        else: raise ValueError('Unknown tuning override: ' + key)
    architecture = cfg['spec']['architecture']
    # Precise absolute/rotary limits are verified again against the actual Phase 1 state.
    cfg['context_limit'] = 4096 if architecture == 'encoder_decoder' else 1024
    assert cfg['max_source_tokens'] > 0 and 2 <= cfg['max_target_tokens'] < cfg['context_limit']
    assert cfg['source_policy'] in ['head', 'tail', 'head_tail', 'spread', 'sentence_mmr']
    assert cfg['rank'] > 0 and cfg['prompt_tokens'] > 0 and cfg['batch_size'] > 0 and cfg['grad_accum'] > 0
    assert cfg['epochs'] > 0 and cfg['encoder_chunk_size'] >= 0
    validate_generation(cfg['generation'])
    return cfg


def validate_generation(policy):
    assert policy['num_beams'] >= 1 and 0 <= policy['min_new_tokens'] < policy['max_new_tokens']
    assert policy['repetition_penalty'] >= 1 and policy['no_repeat_ngram_size'] >= 0
    assert policy['long_document_mode'] in ['select', 'map_reduce'] and policy['map_reduce_max_chunks'] > 0


def run_path(name):
    return PROJECT_DIR / (name + '_' + RUN_TAG)


def resolve_teacher(cfg, student_parameters):
    paths = [Path(cfg['teacher_requested'])] if cfg['teacher_requested'] else [p.parent for p in PROJECT_DIR.glob('*/metrics.json')]
    choices = []
    for path in paths:
        if not (path / 'best_model.pt').is_file() or not (path / 'run_config.json').is_file(): continue
        saved = json.loads((path / 'run_config.json').read_text()); metrics = json.loads((path / 'metrics.json').read_text())
        if saved['data_fingerprint'] != DATA_FINGERPRINT or saved['tokenizer_hash'] != TOKENIZER_HASH: continue
        if saved['metric_config'] != cfg['metric_config'] or saved['spec']['method'] == 'kd': continue
        if not metrics['training_complete'] or not metrics['full_validation'] or metrics['total_parameters'] <= student_parameters: continue
        choices.append((metrics['bertscore_f1'], str(path)))
    if not choices: raise FileNotFoundError('KD requires a completed larger Phase 6 teacher with full held-out evaluation. Run encoder_full first.')
    path = Path(max(choices)[1]); teacher_cfg = json.loads((path / 'run_config.json').read_text())
    if teacher_cfg['phase1_heads'] != PHASE1_HEADS: raise ValueError('Teacher Phase 1 head counts differ.')
    state = torch_load(path / 'best_model.pt'); teacher, metadata = make_model(teacher_cfg)
    teacher.load_state_dict(state['model'], strict=True); del state
    teacher = teacher.to(DEVICE).eval().requires_grad_(False)
    stat = (path / 'best_model.pt').stat()
    return teacher, teacher_cfg, {'path': str(path), 'bytes': stat.st_size, 'mtime_ns': stat.st_mtime_ns,
                                  'teacher_parameters': metadata['total_parameters'], 'mode': 'token_kl_on_train_references_only'}


def representation_report(cfg, output):
    prompt_length = len(tok.encode(cfg['instruction'] + cfg['summary_prefix'], add_special_tokens=False).ids)
    soft = cfg['prompt_tokens'] if cfg['spec']['method'] == 'prompt' else 0
    budget = cfg['max_source_tokens'] if cfg['spec']['architecture'] == 'encoder_decoder' else min(
        cfg['max_source_tokens'], cfg['context_limit'] - cfg['generation']['max_new_tokens'] - soft - prompt_length - 2)
    if budget < 1: raise ValueError('Generation budget leaves no source context.')
    report = {'source_policy': cfg['source_policy'], 'source_budget_at_generation': budget, 'splits': {}}
    for split in DATA:
        lengths = np.asarray([TOKENS[split].index[key][1] for key in DATA[split]])
        row = {'documents': len(lengths), 'source_tokens_p50': float(np.median(lengths)),
               'source_tokens_p90': float(np.quantile(lengths, 0.9)),
               'full_source_within_budget_fraction': float((lengths <= budget).mean()),
               'mean_budget_coverage_upper_bound': float(np.minimum(1.0, budget / np.maximum(lengths, 1)).mean())}
        if split != 'test':
            references = np.asarray([REFERENCES[split].index[key][1] + 1 for key in DATA[split]])
            row.update(reference_tokens_p50=float(np.median(references)), reference_tokens_p90=float(np.quantile(references, 0.9)),
                       reference_truncation_fraction=float((references > cfg['max_target_tokens']).mean()))
        report['splits'][split] = row
    save_json(report, output / 'context_coverage.json')
    print('Context/target length audit:', report)
    return report


def target_loss(logits, labels, smoothing=0.0):
    return F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), labels.reshape(-1),
                           ignore_index=-100, label_smoothing=smoothing)


def kd_loss(student, teacher, labels, temperature):
    per_token = F.kl_div(F.log_softmax(student.float() / temperature, dim=-1),
                          F.softmax(teacher.float() / temperature, dim=-1), reduction='none').sum(-1)
    mask = labels != -100
    return per_token[mask].mean() * temperature ** 2


def evaluate_generation(model, cfg, keys, policy=None):
    model.eval(); policy = policy or cfg['generation']
    rows = generate_rows(model, 'dev', keys, cfg, policy)
    return rows, score_summaries(rows, 'dev', keys)


def run_experiment(name, overrides=None, run_name=None):
    cfg = base_config(name, overrides); run_name = run_name or name; output = run_path(run_name)
    output.mkdir(parents=True, exist_ok=True); config_path = output / 'run_config.json'
    if config_path.is_file() and json.loads(config_path.read_text()) != cfg:
        raise ValueError('Settings changed under an existing run. Use a new RUN_TAG: ' + str(output))
    if not RESUME and (output / 'last.pt').exists(): raise ValueError('Choose a fresh run tag instead of overwriting training.')
    required = ['metrics.json', 'verification.json', 'prediction.json', 'submission.zip', 'best_model.pt', 'last.pt']
    if RESUME and all((output / filename).is_file() for filename in required):
        cached = json.loads((output / 'metrics.json').read_text()); verified = json.loads((output / 'verification.json').read_text())
        checks = ['checkpoint_reload_text_match', 'checkpoint_reload_scores_match', 'json_metric_roundtrip',
                  'one_summary_per_test_document', 'original_ids_preserved', 'zip_verified', 'training_complete']
        if cached.get('training_complete') and all(verified.get(check) is True for check in checks):
            rows = json.loads((output / 'prediction.json').read_text()); validate_rows(rows, 'test', list(DATA['test']))
            with zipfile.ZipFile(output / 'submission.zip') as archive:
                assert archive.namelist() == ['prediction.json'] and json.loads(archive.read('prediction.json')) == rows
            print('Reused verified completed run:', output, '(no retraining or repeated decoding)')
            return cached
    save_json(cfg, config_path); seed_all(cfg['seed'])
    representation_report(cfg, output)
    model, metadata = make_model(cfg); teacher = None; teacher_cfg = None; teacher_info = None
    if cfg['spec']['method'] == 'kd':
        pinned = cfg
        if (output / 'last.pt').is_file() and (output / 'model_metadata.json').is_file():
            previous_teacher = json.loads((output / 'model_metadata.json').read_text()).get('teacher')
            if previous_teacher: pinned = {**cfg, 'teacher_requested': previous_teacher['path']}
        rng = capture_rng()
        try: teacher, teacher_cfg, teacher_info = resolve_teacher(pinned, metadata['total_parameters'])
        finally: restore_rng(rng)
    metadata['teacher'] = teacher_info; save_json(metadata, output / 'model_metadata.json')
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg['spec']['lr'], weight_decay=cfg['weight_decay'])
    keys = list(DATA['train']); batches = math.ceil(len(keys) / cfg['batch_size'])
    steps_per_epoch = math.ceil(batches / cfg['grad_accum']); total_steps = steps_per_epoch * cfg['epochs']
    warmup = max(1, round(total_steps * cfg['warmup_fraction']))
    def schedule(step):
        return (step + 1) / warmup if step < warmup else 0.5 * (1 + math.cos(math.pi * min(1, (step - warmup) / max(1, total_steps - warmup))))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda' and AMP_DTYPE == torch.float16)
    start_epoch = 0; start_batch = 0; step = 0; best = -float('inf'); bad = 0; complete = False; history = []; elapsed = 0.0
    if RESUME and (output / 'last.pt').is_file():
        saved = torch_load(output / 'last.pt')
        if saved['config'] != cfg or saved['teacher'] != teacher_info: raise ValueError('Saved config/teacher signature differs.')
        model.load_state_dict(saved['model'], strict=True); optimizer.load_state_dict(saved['optimizer'])
        scheduler.load_state_dict(saved['scheduler']); scaler.load_state_dict(saved['scaler']); restore_rng(saved['rng'])
        start_epoch = saved['epoch']; start_batch = saved['next_batch']; step = saved['step']; best = saved['best']; bad = saved['bad']
        history = saved['history']; elapsed = saved['seconds']; complete = saved['complete']; del saved
    start = time.time()
    if DEVICE.type == 'cuda': torch.cuda.reset_peak_memory_stats()
    epoch_keys = fixed_validation_keys(cfg['epoch_validation_limit'])
    save_json({'keys': epoch_keys, 'scope': 'fixed held-out subset for checkpoint selection'}, output / 'selection_manifest.json')
    def persist(epoch, next_batch):
        save_torch({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                    'scaler': scaler.state_dict(), 'rng': capture_rng(), 'config': cfg, 'teacher': teacher_info,
                    'epoch': epoch, 'next_batch': next_batch, 'step': step, 'best': best, 'bad': bad,
                    'history': history, 'seconds': elapsed + time.time() - start, 'complete': complete}, output / 'last.pt')
        save_json(history, output / 'history.json')
    if not complete:
        for epoch in range(start_epoch, cfg['epochs']):
            if cfg['max_optimizer_steps'] is not None and step >= cfg['max_optimizer_steps']: break
            order = list(keys); random.Random(cfg['seed'] + epoch).shuffle(order)
            begin = start_batch if epoch == start_epoch else 0; next_batch = begin; capped = False
            model.train(); optimizer.zero_grad(set_to_none=True); accumulated = 0.0
            progress = tqdm(range(begin, batches), desc=f'{run_name} epoch {epoch + 1}/{cfg["epochs"]}', unit='batch')
            for i in progress:
                current = order[i * cfg['batch_size']:(i + 1) * cfg['batch_size']]
                batch = pack_batch('train', current, cfg)
                # Normalize by all supervised tokens in this accumulation group, including the short last group.
                group_start = (i // cfg['grad_accum']) * cfg['grad_accum']
                group_keys = order[group_start * cfg['batch_size']:min(batches, group_start + cfg['grad_accum']) * cfg['batch_size']]
                total_tokens = sum(min(len(REFERENCES['train'][key]) + 1, cfg['max_target_tokens']) for key in group_keys)
                token_fraction = int((batch['labels'] != -100).sum()) / total_tokens
                with amp_context():
                    logits = model(batch); loss = target_loss(logits, batch['labels'], cfg['label_smoothing'])
                    if teacher is not None:
                        teacher_batch = pack_batch('train', current, teacher_cfg, batch['targets_list'])
                        with torch.no_grad(): teacher_logits = teacher(teacher_batch)
                        loss = (1 - cfg['kd_weight']) * loss + cfg['kd_weight'] * kd_loss(logits, teacher_logits, batch['labels'], cfg['kd_temperature'])
                if not torch.isfinite(loss): raise FloatingPointError('Nonfinite summarization loss.')
                scaler.scale(loss * token_fraction).backward(); accumulated += loss.item() * token_fraction; next_batch = i + 1
                progress.set_postfix(loss=f'{loss.item():.4f}', step=step)
                if next_batch % cfg['grad_accum'] == 0 or next_batch == batches:
                    scaler.unscale_(optimizer); nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                    previous = scaler.get_scale(); scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
                    if scaler.get_scale() >= previous:
                        scheduler.step(); step += 1
                        history.append({'kind': 'train', 'epoch': epoch + 1, 'step': step, 'loss': accumulated})
                        if step % SAVE_EVERY_STEPS == 0: persist(epoch, next_batch)
                    accumulated = 0.0
                    if cfg['max_optimizer_steps'] is not None and step >= cfg['max_optimizer_steps']: capped = True; break
            progress.close()
            rows, metrics = evaluate_generation(model, cfg, epoch_keys)
            metrics['validation_loss'] = validation_loss(model, cfg)
            history.append({'kind': 'dev', 'epoch': epoch + 1, 'step': step, **metrics})
            show_scores(metrics, 'Fixed validation subset — epoch ' + str(epoch + 1))
            if metrics['bertscore_f1'] > best + 1e-12:
                best = metrics['bertscore_f1']; bad = 0
                save_torch({'model': model.state_dict(), 'config': cfg, 'metadata': metadata, 'epoch': epoch + 1}, output / 'best_model.pt')
                save_json(metrics, output / 'best_selection_metrics.json'); save_json(rows, output / 'best_selection_prediction.json')
            else: bad += 1
            finished = next_batch == batches
            complete = finished and (epoch + 1 == cfg['epochs'] or (cfg['patience'] > 0 and bad >= cfg['patience']))
            persist(epoch + 1 if finished else epoch, 0 if finished else next_batch)
            if complete or capped: break
            model.train()
    if teacher is not None: del teacher; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    if not (output / 'best_model.pt').is_file(): raise RuntimeError('No evaluated checkpoint exists.')
    state = torch_load(output / 'best_model.pt'); model.load_state_dict(state['model'], strict=True); best_epoch = state['epoch']; del state
    verify_rows, verified = evaluate_generation(model, cfg, epoch_keys)
    assert verify_rows == json.loads((output / 'best_selection_prediction.json').read_text()), 'Reloaded summaries differ from the selected checkpoint.'
    recorded = json.loads((output / 'best_selection_metrics.json').read_text())
    for metric in ['bertscore_f1', 'rouge1_f1', 'rougeL_f1']:
        assert math.isclose(verified[metric], recorded[metric], rel_tol=0, abs_tol=1e-6), 'Reloaded metrics differ.'
    final_keys = fixed_validation_keys(cfg['final_validation_limit'])
    if final_keys == epoch_keys: final_rows, metrics = verify_rows, verified
    else: final_rows, metrics = evaluate_generation(model, cfg, final_keys)
    save_json(final_rows, output / 'validation_prediction.json')
    saved_rows = json.loads((output / 'validation_prediction.json').read_text())
    roundtrip = score_summaries(saved_rows, 'dev', final_keys)
    for metric in ['bertscore_f1', 'rouge1_f1', 'rougeL_f1']:
        assert math.isclose(roundtrip[metric], metrics[metric], abs_tol=1e-6)
    metrics.update({'experiment': name, 'run_name': run_name, 'architecture': cfg['spec']['architecture'], 'method': cfg['spec']['method'],
                    'best_epoch': best_epoch, 'training_complete': complete, 'full_validation': len(final_keys) == len(DATA['dev']),
                    'optimizer_steps': step, 'training_seconds': elapsed + time.time() - start,
                    'peak_gpu_memory_bytes': torch.cuda.max_memory_allocated() if DEVICE.type == 'cuda' else 0, **metadata})
    save_json(metrics, output / 'metrics.json'); save_json(cfg['generation'], output / 'generation_config.json')
    show_scores(metrics, 'Best checkpoint — final held-out evaluation')
    test_rows = generate_rows(model, 'test', list(DATA['test']), cfg, cfg['generation']); save_submission(output, test_rows)
    save_json({'checkpoint_reload_text_match': True, 'checkpoint_reload_scores_match': True, 'json_metric_roundtrip': True,
               'one_summary_per_test_document': True, 'original_ids_preserved': True, 'zip_verified': True,
               'training_complete': complete, 'full_validation': metrics['full_validation'], 'private_test_score': None}, output / 'verification.json')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4)); tr = [r for r in history if r['kind'] == 'train']; dv = [r for r in history if r['kind'] == 'dev']
    axes[0].plot([r['step'] for r in tr], [r['loss'] for r in tr]); axes[0].set(xlabel='Optimizer step', ylabel='Target loss')
    axes[1].plot([r['epoch'] for r in dv], [r['bertscore_f1'] for r in dv]); axes[1].set(xlabel='Epoch', ylabel='Validation BERTScore F1')
    fig.tight_layout(); fig.savefig(output / 'training_curves.png', dpi=130); plt.show(); plt.close(fig)
    print('Verification PASS. Saved:', output / 'submission.zip')
    del model, optimizer; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return metrics


def search_decoding(name):
    source = run_path(name); cfg = json.loads((source / 'run_config.json').read_text())
    model, _ = make_model(cfg); state = torch_load(source / 'best_model.pt'); model.load_state_dict(state['model'], strict=True); del state
    model = model.to(DEVICE).eval(); keys = fixed_validation_keys(cfg['epoch_validation_limit'])
    _, baseline = evaluate_generation(model, cfg, keys); best_score = baseline['bertscore_f1']; best_policy = cfg['generation']; trials = []
    for trial in tqdm(DECODING_CANDIDATES, desc='Validation decoding search', unit='policy'):
        policy = {**cfg['generation'], **trial['overrides']}; validate_generation(policy)
        _, metrics = evaluate_generation(model, cfg, keys, policy); trials.append({'name': trial['name'], 'policy': policy, 'metrics': metrics})
        if metrics['bertscore_f1'] > best_score: best_score = metrics['bertscore_f1']; best_policy = policy
    output = run_path(name + '__decoding'); output.mkdir(exist_ok=True)
    save_json({'source': str(source), 'selection_scope': keys, 'selected_policy': best_policy, 'trials': trials}, output / 'decoding_search.json')
    final_keys = fixed_validation_keys(cfg['final_validation_limit'])
    rows, metrics = evaluate_generation(model, cfg, final_keys, best_policy); save_json(rows, output / 'validation_prediction.json'); save_json(metrics, output / 'metrics.json')
    save_submission(output, generate_rows(model, 'test', list(DATA['test']), cfg, best_policy))
    del model; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return metrics

## 9. Preflight invariants

In [ ]:
assert set(FILES)=={'train','test'}
assert all(row['summary'] is None for row in DATA['test'].values())
assert set(DATA['train']).isdisjoint(DATA['dev'])
assert not ({document_hash(row) for row in DATA['train'].values()} & {document_hash(row) for row in DATA['dev'].values()})
base=nn.Linear(7,11); x=torch.randn(2,3,7)
for multiplicative in [False,True]:
    adapter=LoRAProjection(copy.deepcopy(base),3,6,multiplicative)
    torch.testing.assert_close(adapter(x),base(x))
    adapter(x).sum().backward()
    assert adapter.B.weight.grad.abs().sum()>0 and adapter.base.weight.grad is None
assert clean_summary('Court allowed. Court allowed.',True)=='Court allowed.'
print('PASS: grouped split, private-test label isolation, LoRA/LoRMA identity and trainable gradients.')

## 10. Run selected/core/extra profiles and optional training/decoding searches

In [ ]:
selected=list(CORE_EXPERIMENTS) if RUN_ALL else [EXPERIMENT]
if RUN_EXTRA_APPROACHES: selected=list(dict.fromkeys(selected+list(EXTRA_EXPERIMENTS)))
results=[]
for name in tqdm(selected,desc='Phase 6 experiments',unit='run'):
    results.append(run_experiment(name))
if RUN_TUNING:
    for trial in tqdm(TUNING_TRIALS,desc='Training trials',unit='trial'):
        results.append(run_experiment(TUNE_BASE_EXPERIMENT,trial['overrides'],TUNE_BASE_EXPERIMENT+'__'+trial['name']))
if RUN_DECODING_SEARCH:
    decoding_metrics=search_decoding(EXPERIMENT)

## 11. Compare all compatible runs and required architecture/method coverage

In [ ]:
import csv
records = []
for file in PROJECT_DIR.glob('*/metrics.json'):
    folder = file.parent
    if not (folder / 'run_config.json').is_file(): continue
    cfg = json.loads((folder / 'run_config.json').read_text())
    if cfg['data_fingerprint'] != DATA_FINGERPRINT or cfg['tokenizer_hash'] != TOKENIZER_HASH or cfg['metric_config'] != metric_config(): continue
    row = json.loads(file.read_text()); row['path'] = str(folder); records.append(row)
completed = [row for row in records if row['training_complete'] and row['full_validation']]
methods = {row['method'] for row in completed}; origins = {row['architecture'] for row in completed}
digests = []
best_per_origin = {}
for row in sorted(completed, key=lambda item: item['bertscore_f1'], reverse=True):
    folder = Path(row['path']); prediction = json.loads((folder / 'prediction.json').read_text())
    validate_rows(prediction, 'test', list(DATA['test']))
    digests.append(hashlib.sha256(json.dumps(prediction, sort_keys=True).encode()).hexdigest())
    best_per_origin.setdefault(row['architecture'], {'run': row['run_name'], 'bertscore_f1': row['bertscore_f1'], 'submission': str(folder / 'submission.zip')})
coverage = {'four_completed_experiments': len(completed) >= 4, 'six_completed_experiments': len(completed) >= 6,
            'six_distinct_prediction_contents': len(set(digests)) >= 6, 'all_four_phase1_origins': origins >= set(MODEL_CLASSES),
            'three_non_kd_adaptations': len(methods - {'kd'}) >= 3,
            'four_techniques_listed_in_phase6': len(methods & {'lora', 'lorma', 'houlsby', 'pfeiffer', 'prompt', 'kd'}) >= 4,
            'compulsory_lorma': 'lorma' in methods, 'compulsory_kd': 'kd' in methods,
            'competition_acceptance_verified': False}
save_json(coverage, PROJECT_DIR / 'coverage.json'); save_json(best_per_origin, PROJECT_DIR / 'best_per_architecture.json')
columns = ['run_name', 'architecture', 'method', 'bertscore_f1', 'rouge1_f1', 'rougeL_f1', 'best_epoch', 'training_seconds',
           'total_parameters', 'trainable_parameters', 'peak_gpu_memory_bytes', 'training_complete', 'full_validation', 'path']
with (PROJECT_DIR / 'experiment_summary.csv').open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=columns); writer.writeheader()
    for row in sorted(records, key=lambda item: item['bertscore_f1'], reverse=True):
        writer.writerow({key: row.get(key) for key in columns})
        print(row['run_name'], 'validation BERTScore=', round(row['bertscore_f1'], 6), 'complete=', row['training_complete'])
print('Required experiment coverage:', coverage)
print('Best per Phase 1 architecture:', best_per_origin)

## 12. How to tune and run in Colab

All controls are in **Section 3**. Start with one `encoder_full` run. Set up a Colab GPU, an `HF` secret
containing your plain `hf_...` token, and `MyDrive/cs781_phase1/` with all four final Phase 1 checkpoints
and the BPE tokenizer. This notebook is standalone; no Phase 2–5 training is rerun.

In an already prepared runtime, changing only the experiment requires rerunning **Section 3, then
Sections 10–11**. A new runtime must run all cells. Changing `SEED`, `VALIDATION_FRACTION` or overlap
filtering requires rebuilding Section 4 as well; rerun all cells for those dataset changes. Use fresh
tags for manual changes. Changing BERTScore model/layers/rescaling rebuilds its cached metric model.

| Goal | Control |
|---|---|
| One experiment | `EXPERIMENT='encoder_full'`, `RUN_ALL=False` |
| Six core experiments | `RUN_ALL=True`; larger teachers finish before KD |
| Ten optional profiles | `RUN_EXTRA_APPROACHES=True` |
| Automatic training sweep | `RUN_TUNING=True`, edit `TUNING_TRIALS`, set `TUNE_BASE_EXPERIMENT` |
| Dev-only generation sweep | `RUN_DECODING_SEARCH=True` after the selected model has finished |
| Fresh manual trial | New `RUN_TAG`; `LEARNING_RATE` overrides the profile LR |
| Resume interrupted training | Identical tag/settings; `RESUME=True` |
| Epochs/overfitting | `EPOCHS`, `EARLY_STOPPING_PATIENCE`; select by held-out BERTScore, not training loss |
| Case batch and accumulation | `BATCH_SIZE`, `GRAD_ACCUM_STEPS`; loss weighting counts actual supervised tokens |
| Input and target lengths | `MAX_SOURCE_TOKENS`, `MAX_TARGET_TOKENS` |
| Source selection | `SOURCE_POLICY`: head/tail/head_tail/spread/sentence_mmr |
| Native encoder hierarchy | `ENCODER_CHUNK_SIZE=384` independently encodes source windows and combines their memories |
| LoRA/LoRMA capacity | `RANK`, `ALPHA` |
| Adapter capacity | `BOTTLENECK_DIM` |
| Soft prompt capacity | `PROMPT_TOKENS`; frozen backbone, learned continuous prefix |
| Partial tuning depth | `PARTIAL_LAYERS` |
| Teacher/student transfer | `KD_WEIGHT`, `KD_TEMPERATURE`, `TEACHER_RUN_DIR` |
| Generated length | `GENERATION['max_new_tokens']`, `['min_new_tokens']` — tokens, not words |
| Greedy vs beam | `GENERATION['num_beams']=1` vs 2/3 |
| Length/repetition | `length_penalty`, `repetition_penalty`, `no_repeat_ngram_size` |
| Redundant sentences | `GENERATION['deduplicate_sentences']=True` |
| Two-stage generation | `GENERATION['long_document_mode']='map_reduce'`, `map_reduce_max_chunks` |
| Validation fraction | `VALIDATION_FRACTION`, `SEED`; keep fixed for comparable experiments |
| Missing/unusable training fields | `INVALID_TRAIN_ROW_POLICY='exclude'` audits/excludes them; `error` stops |
| Unexpected invalid-record rate | `MAX_INVALID_TRAIN_FRACTION=0.05`; inspect data_quality_report.json if exceeded |
| Epoch selection runtime | `EPOCH_VALIDATION_LIMIT=128`; 0 uses all held-out documents each epoch |
| Final metric verification | Keep `FINAL_VALIDATION_LIMIT=0` to score every held-out document |
| Validation loss runtime | `EVAL_LOSS_LIMIT`; diagnostic only, not a replacement for generation metrics |
| BERTScore settings | `BERTSCORE_MODEL`, `BERTSCORE_LAYERS`, `BERTSCORE_RESCALE`; match any published official scorer |
| Metric runtime/memory | `BERTSCORE_BATCH_SIZE`, `BERTSCORE_DEVICE='cpu'` or `'cuda'` |

Core profiles: `encoder_full`, `decoder_lora`, `hybrid_lorma`, `encoder_adapters`,
`lightweight_prompt`, `lightweight_kd`. They cover all four origins, compulsory LoRMA/KD and at least
four techniques explicitly listed in Phase 6. The PDF has conflicting global submission minimums
(six in general guidelines, four in §6.5/§12); six substantially different core runs cover the stricter count.
Check accepted and distinct submissions on Codabench yourself; local ZIPs do not establish acceptance.

Optional profiles: `decoder_full`, `encoder_lora`, `encoder_lorma`, `encoder_partial`, `hybrid_lora`,
`hybrid_adapters`, `lightweight_lorma`, `lightweight_lora`, `lightweight_pfeiffer`, `lightweight_full`.
Full/partial fine-tuning are useful baselines in addition to the techniques explicitly listed in §12.2.1.
The adapter implementations use the described residual sites in your existing pre-norm architecture;
the Pfeiffer-style profile has one output bottleneck per layer.

Run individual experiments in separate Colab notebook copies with `RUN_ALL=False`,
`RUN_EXTRA_APPROACHES=False`, `RUN_TUNING=False`, `RUN_DECODING_SEARCH=False`. Prepare Section 4 once
before other sessions use shared caches. Do not run the same experiment/tag concurrently. Run KD after a
larger Phase 6 teacher completes. Changing a manual training/decoding setting requires a fresh tag;
the decoding search uses a separate `<experiment>__decoding_<tag>` directory without retraining.

The training sweep really trains each listed trial and saves overrides under separate names. Trials
are one-factor changes, not cumulative. Choose the strongest measured base and shorten the list to fit
your GPU budget. Source/target budgets share the decoder-only context; longer summaries reduce its
available source context. The native encoder–decoder has separate source and target context budgets.
Generation never silently discards the beginning of a generated summary through a rolling window.

`head_tail`/`spread`/MMR are selective representations, not full document reading. `sentence_mmr` ranks
source sentences by document-centrality and redundancy without consulting reference summaries.
Map–reduce reads the selected chunk budget and performs several extra generation calls; compare on
the same validation set before submitting. Native encoder chunking changes attention scope while
retaining selected source tokens in cross-attention memory. Increasing budgets, beams or epochs is not
guaranteed to improve quality and costs more runtime.

**Distillation:** the student retains alternating Phase 1 lightweight layers. A larger trained Phase 6
teacher supplies token-level distributions on **train references only**. Both models use the same BPE
and target sequence; their document representations can differ. The teacher is frozen, not updated by
student gradients. No validation summaries become training labels. Expect extra training compute and
memory for two models; reduce `BATCH_SIZE` and increase accumulation if necessary. This is online
token-KL distillation; no sequence-cache or external teacher is silently substituted.

**Scores:** checkpoint selection scores are from a fixed held-out subset by default. Final metrics are
from the full held-out split, which includes those selection cases and is not an untouched independent
test. The best subset epoch may differ from the best hypothetical full-validation epoch. Use full epoch
validation (`EPOCH_VALIDATION_LIMIT=0`) for closer model selection when compute permits.
All runs must use the same split and metric settings. BERTScore/ROUGE do not prove factual accuracy:
read summaries and inspect entities, dates, provisions, holdings, coverage and repetition yourself.

The official PDF does not specify scorer version, rescaling, layers or ROUGE stemming. Defaults are
explicit and saved with BERTScore's hash, not claimed to reproduce an unseen scorer. RoBERTa-large is
downloaded **only as a metric**; the summarizer and KD teacher use Phase 1 weights exclusively. Do not
change the metric model just to raise numbers. Long summaries may be truncated by the metric model's
tokenizer; keep evaluation settings consistent and compare realistic output lengths.

Drive outputs: `MyDrive/cs781_phase6/<experiment>_<RUN_TAG>/`. Keep `best_model.pt` for inference/teachers,
`last.pt` for resume, and all configurations/metrics/predictions/verification files for your report.
`prediction.json` is a list of `{id, summary}` objects; `submission.zip` contains only that JSON at its root.
`best_per_architecture.json` and `experiment_summary.csv` compare compatible complete runs. Retain the
required architecture-specific and compulsory-method submissions even if another run scores better.
Actual private test scores are available only from Codabench. Keep the required LLM declaration.

References: capstone PDF §12 and professor's Phase 1 starter for workflow; your Phase 1 model
definitions and checkpoint inference; Phase 5's robust input/Drive/RNG handling.

- [BERTScore implementation and reproducibility settings](https://github.com/Tiiiger/bert_score).
- [ROUGE implementation](https://github.com/google-research/google-research/tree/master/rouge).
- [LoRA](https://arxiv.org/abs/2106.09685), [LoRMA](https://aclanthology.org/2025.findings-acl.527/).
- [Houlsby adapters](https://arxiv.org/abs/1902.00751), [prompt tuning](https://arxiv.org/abs/2104.08691).

Adapter placement uses bottlenecks at the stated residual sites in the existing pre-norm Phase 1 blocks;
it is not a claim of reproducing every detail of the original papers. Keep the required LLM declaration.
Only actual generated validation/test results support performance conclusions; implementation tests do not.